# 2° Etapa do Projeto Prático


## Parte 1 - Dados

Amostragem de dados aleatórios, rotulação humana e pré-processamento dos dados.

### 1.1 - Importação dos Dados

In [ ]:
!pip install emoji --upgrade

In [ ]:
import gdown
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import emoji
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
# Download dos comentários
file_id = '1t7gDsh20kp-5E2cxAG0X6Dcs2_PVWIRh'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'posts_AnsiedadeDepressao.json', quiet=False)

file_id = '1LZ0MbbZR2vDuCx-25fHHx13MgClLH-vV'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'comentarios_AnsiedadeDepressao.json', quiet=False)

file_id = '1Tdyp7LMwH9xluteGrThz-pTf8f9eBI7b'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'posts_desabafos.json', quiet=False)

file_id = '179Kf10ELpVwU8sL2ZIBu0-WtvREG7W7Q'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'comentarios_desabafos.json', quiet=False)

### 1.2 - Normalização e Filtragem dos Dados
Remove:
- Comentários do autor do post;
- Comentários de resposta a comentários que não sejam do autor do post;
- Comentários removidos peol usuário, reddit ou ferramentas externas (redact);
- Comentários de tamanho insuficiente (<3).

Normalização:
- Converte emojis pra texto
- Substitui '\n' por espaços
- Remove múltiplos espaços

In [ ]:
def normalizar(texto):
    if not texto:
        return ""

    # Remove "Espaço de Largura Zero" do Reddit (&amp;#x200B;)
    texto = texto.replace("&amp;#x200B;", " ")

    # Substituir quebras de linha por espaço
    texto = texto.replace("\n", " ").replace("\r", " ")

    # Limpar marcações de formatação do Markdown do Reddit
    texto = re.sub(r'\*\*|\*|~~|`|>!|!<|_', ' ', texto) # Remove asteriscos, underlines, tios duplos, crases e os marcadores de spoiler/citação (>!)
    texto = re.sub(r'^\s*>\s*', ' ', texto) # Remove o '>' de citações no início do texto

    # Converter emojis para texto em português
    texto = emoji.demojize(texto, language='pt')

    # Remover múltiplos espaços em branco
    texto = re.sub(r'\s+', ' ', texto).strip()

    if not texto.strip():
        return "[texto_vazio]"

    return texto

In [ ]:
def filtra_comentarios(ARQ_POSTS, ARQ_COMENTARIOS, ARQ_SAIDA, COMMENT_MIN_TAM, LIMITE_COMENTARIOS):
    # Carrega arquivos
    with open(ARQ_POSTS, "r", encoding="utf-8") as f:
        posts = json.load(f)
    with open(ARQ_COMENTARIOS, "r", encoding="utf-8") as f:
        comentarios = json.load(f)

    post_author = {p["id"]: p["author"] for p in posts} # Salva os autores dos posts

    # Filtragem
    limite_atingido = False
    post_limite = None
    comentarios_mantidos_count = 0
    textos_vistos = set()

    comentarios_do_autor_do_post = set()
    for c in comentarios:
        link_id = c.get("link_id", "")
        post_id = link_id.replace("t3_", "")
        autor_comentario = c.get("author")
        autor_post = post_author.get(post_id)

        if autor_comentario == autor_post:
            comentarios_do_autor_do_post.add(c.get("name"))

    for c in comentarios:
        c['filtered'] = None

        # Pega o link do comentário e remove o prefixo para obter o id do post
        link_id = c.get("link_id", "")
        parent_id = c.get("parent_id", "")
        post_id = link_id.replace("t3_", "")

        # Verifica se já passou do limite
        # Continua até o final do post
        if limite_atingido:
            if post_id != post_limite:
                break  # saiu do post -> pode parar

        # Pega autor e texto do comentário
        autor_comentario = c.get("author")
        texto = c.get("body", "")

        autor_post = post_author.get(post_id)

        # Regras de Filtragem

        # Se o comentário for do autor do post
        if autor_post and autor_comentario == autor_post:
            c['filtered'] = "is_author"
        # Se o comentário não for uma resposta direta ao post, nem uma resposta a um comentário do autor do post
        elif parent_id != link_id and parent_id not in comentarios_do_autor_do_post:
            c['filtered'] = "answer_not_author"
        # Se for do automoderador
        elif autor_comentario == "AutoModerator":
            c['filtered'] = "automoderator"
        else:
            c['body'] = normalizar(c.get("body", "")) # Normaliza o texto
            texto = c['body']

            # Se o comentário tiver sido censurado pela ferramenta Redact
            if "[redact]" in texto.lower():
                c['filtered'] = "redact"
            # Se o comentário tiver sido removido pelo reddit ou usuário
            elif "[removed]" in texto.lower() or "[ removed by reddit ]" in texto.lower():
                c['filtered'] = "removed"
            # Se for um comentário de bot
            elif "i am a bot" in texto.lower():
                c['filtered'] = "bot"
            # Se o comentário for muito pequeno
            elif len(texto.split()) < COMMENT_MIN_TAM:
                c['filtered'] = "tamanho_insuficiente"
            # Se o comentário for duplicado
            elif texto in textos_vistos:
                c['filtered'] = "duplicado"

        # Se o comentário passar todos os filtros...
        if c['filtered'] is None:
            textos_vistos.add(texto) # Adiciona a lista de duplicados
            comentarios_mantidos_count += 1 # Aumenta a contagem
            if comentarios_mantidos_count >= LIMITE_COMENTARIOS:
                limite_atingido = True
                post_limite = post_id

    # Filtra apenas os comentários que passaram todos od filtros
    comentarios_filtrados = [c for c in comentarios if c.get("filtered") is None]

    # Salva no arquivo de saída
    with open(ARQ_SAIDA, "w", encoding="utf-8") as f:
        json.dump(comentarios_filtrados, f, ensure_ascii=False, indent=4)

    print(f"Total de comentários: {len(comentarios)}")

    counter = {}
    for c in comentarios:
        counter[c.get("filtered")] = counter.get(c.get("filtered"), 0) + 1
    print(counter)

    return comentarios

In [ ]:
COMMENT_MIN_TAM = 3
LIMITE_COMENTARIOS = 99999  # limite de comentários

In [ ]:
#AnsiedadeDepressao

# Caminhos
ARQ_POSTS = "posts_AnsiedadeDepressao.json"
ARQ_COMENTARIOS = "comentarios_AnsiedadeDepressao.json"
ARQ_SAIDA = "comentarios_AnsiedadeDepressao_filtrados.json"

comentarios_processados = []

comentarios_processados.extend(filtra_comentarios(ARQ_POSTS, ARQ_COMENTARIOS, ARQ_SAIDA, COMMENT_MIN_TAM, LIMITE_COMENTARIOS))
print(f"\nQuantia de Não Filtrados: {len(comentarios_processados)}")

In [ ]:
#Desabafos

# Caminhos
ARQ_POSTS = "posts_desabafos.json"
ARQ_COMENTARIOS = "comentarios_desabafos.json"
ARQ_SAIDA = "comentarios_desabafos_filtrados.json"

comentarios_processados.extend(filtra_comentarios(ARQ_POSTS, ARQ_COMENTARIOS, ARQ_SAIDA, COMMENT_MIN_TAM, LIMITE_COMENTARIOS))
print(f"\nQuantia de Não Filtrados: {len(comentarios_processados)}")

In [ ]:
with open("comentarios_AnsiedadeDepressao_filtrados.json", "r", encoding="utf-8") as f:
    a = json.load(f)

with open("comentarios_desabafos_filtrados.json", "r", encoding="utf-8") as f:
    b = json.load(f)

resultado = a + b
print(len(resultado))

with open("corpus.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, ensure_ascii=False, indent=4)

In [ ]:
for c in resultado:
    if c['body'] == 'Caramba. Talvez esse tenha sido o relato mais absurdo que já vi nesse sub. Absurdo ao ponto de eu não querer te ajudar. Você é um merda. Tem que se foder mesmo. Ou se mata e acaba com essa merda, ou se interna e tenta voltar pra sociedade daqui uns 10 anos, no mínimo.':
        print(c)

In [ ]:
# 1. CONFIGURAÇÕES VISUAIS E ALTA RESOLUÇÃO
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid")

# 2. PREPARAÇÃO ÚNICA E OTIMIZADA DOS DADOS
df_plots = pd.DataFrame(comentarios_processados)

# Criação de métricas de controle de status
df_plots['status'] = df_plots['filtered'].apply(lambda x: 'Mantido' if pd.isna(x) else 'Excluído')
df_plots['motivo_filtro'] = df_plots['filtered'].fillna('Mantido (Aprovado)')
df_plots['contagem_palavras'] = df_plots['body'].apply(lambda x: len(str(x).split()) if x else 0)

# Tratamento de datas e fuso horário (UTC -> América/São_Paulo)
df_plots['datetime'] = (pd.to_datetime(df_plots['created_utc'], unit='s')
                        .dt.tz_localize('UTC')
                        .dt.tz_convert('America/Sao_Paulo')
                        .dt.tz_localize(None))
df_plots['hora'] = df_plots['datetime'].dt.hour
df_plots['ano_mes'] = df_plots['datetime'].dt.to_period('M').astype(str)
df_plots['dia_semana'] = df_plots['datetime'].dt.day_name().map({
    'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta',
    'Thursday': 'Quinta', 'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
})
ordem_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']

# Subconjunto filtrado (Apenas comentários aprovados/mantidos)
df_mantidos = df_plots[df_plots['status'] == 'Mantido'].copy()
if not df_mantidos.empty:
    df_mantidos['tipo_resposta'] = df_mantidos.apply(
        lambda r: 'Resposta Direta ao Post' if r['parent_id'] == r['link_id'] else 'Resposta ao Autor (Thread)',
        axis=1
    )

# 3. CRIAÇÃO DA SUPER GRADE DO DASHBOARD (4 linhas x 2 colunas)
fig, axes = plt.subplots(4, 2, figsize=(16, 24))

# --- [0, 0] G1: Funil de Filtragem ---
order_motivos = df_plots['motivo_filtro'].value_counts().index
ax1 = sns.countplot(data=df_plots, y='motivo_filtro', order=order_motivos, hue='motivo_filtro', palette='Set2', ax=axes[0, 0], legend=False)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%d', padding=5)
axes[0, 0].set_title('Diagnóstico da Pipeline: Destino dos Comentários', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Quantidade de Comentários')
axes[0, 0].set_ylabel('Motivo/Status')

# --- [0, 1] G2: Proporção por Subreddit ---
sns.countplot(data=df_plots, x='subreddit', hue='status', palette=['#2ecc71', '#e74c3c'], ax=axes[0, 1])
axes[0, 1].set_title('Proporção de Comentários Mantidos vs Excluídos por Subreddit', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Subreddit')
axes[0, 1].set_ylabel('Quantidade de Comentários')
axes[0, 1].legend(title='Status do Filtro')

# --- [1, 0] G5: Distribuição Geral do Tamanho ---
sns.histplot(data=df_mantidos, x='contagem_palavras', kde=True, color='#9b59b6', bins=30, ax=axes[1, 0])
axes[1, 0].set_title('Distribuição do Tamanho dos Comentários Aprovados', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Número de Palavras')
axes[1, 0].set_ylabel('Frequência')
if not df_mantidos.empty:
    axes[1, 0].set_xlim(0, df_mantidos['contagem_palavras'].quantile(0.95))

# --- [1, 1] G6: Boxplot comparativo entre Subreddits ---
sns.boxplot(data=df_mantidos, x='subreddit', y='contagem_palavras', hue='subreddit', palette='Set2', showfliers=False, ax=axes[1, 1], legend=False)
axes[1, 1].set_title('Comprimento dos Comentários por Comunidade (Sem Outliers)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Subreddit')
axes[1, 1].set_ylabel('Quantidade de Palavras')

# --- [2, 0] G7: Tipo de Resposta (Pizza) ---
if not df_mantidos.empty:
    tipo_counts = df_mantidos['tipo_resposta'].value_counts()
    axes[2, 0].pie(tipo_counts, labels=tipo_counts.index, autopct='%1.1f%%', startangle=90, colors=['#3498db', '#f1c40f'], wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[2, 0].set_title('Composição do Dataset Final: Estrutura das Respostas', fontsize=12, fontweight='bold')

# --- [2, 1] G8: Evolução Temporal ---
df_linha = df_plots.groupby(['ano_mes', 'status']).size().unstack(fill_value=0)
df_linha.plot(kind='line', marker='o', color=['#e74c3c', '#2ecc71'], ax=axes[2, 1], linewidth=2)
axes[2, 1].set_title('Evolução do Volume de Comentários ao Longo do Tempo', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Período (Ano-Mês)')
axes[2, 1].set_ylabel('Quantidade de Comentários')
axes[2, 1].tick_params(axis='x', rotation=45)
axes[2, 1].legend(title='Status')

# --- [3, 0] G9: Volumetria por Hora do Dia ---
df_hora = pd.crosstab(df_plots['hora'], df_plots['status'])
df_hora.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], ax=axes[3, 0], width=0.8)
axes[3, 0].set_title('Atividade por Hora do Dia (Horário de Brasília)', fontsize=12, fontweight='bold')
axes[3, 0].set_xlabel('Hora do Dia (0h - 23h)')
axes[3, 0].set_ylabel('Quantidade de Comentários')
axes[3, 0].tick_params(axis='x', rotation=0)
axes[3, 0].legend(title='Status')

# --- [3, 1] G10: Distribuição por Dia da Semana ---
sns.countplot(data=df_plots, x='dia_semana', hue='status', order=ordem_dias, palette=['#2ecc71', '#e74c3c'], ax=axes[3, 1])
axes[3, 1].set_title('Volume de Comentários por Dia da Semana', fontsize=12, fontweight='bold')
axes[3, 1].set_xlabel('Dia da Semana')
axes[3, 1].set_ylabel('Quantidade de Comentários')
axes[3, 1].legend(title='Status')

# Ajusta automaticamente os espaçamentos para não sobrepor textos e labels
plt.tight_layout()
plt.show()
plt.close('all')

### 1.3 - Amostragem

In [ ]:
N_AMOSTRAS = 3000

# Carrega o corpus normalizado do JSON
with open("corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

# Converte para DataFrame
df = pd.DataFrame(corpus)

# Calcula a proporção exata para a amostra
total_dados = len(df)
proporcao_amostra = N_AMOSTRAS / total_dados

# 4. Faz a divisão estratificada baseada na coluna escolhida
df_restante, df_amostrado = train_test_split(
    df,
    test_size=proporcao_amostra,
    random_state=345345,
    stratify=df['subreddit']
)

# Converte de volta para dicionário/lista do Python para salvar em JSON
corpus_amostrado = df_amostrado.to_dict(orient="records")
corpus_restante = df_restante.to_dict(orient="records")

# Salva as duas partes nos arquivos correspondentes
with open("corpus_amostrado.json", "w", encoding="utf-8") as f:
    json.dump(corpus_amostrado, f, ensure_ascii=False, indent=4)

with open("corpus_restante.json", "w", encoding="utf-8") as f:
    json.dump(corpus_restante, f, ensure_ascii=False, indent=4)

print(f"Amostra: {len(corpus_amostrado)} | Restante: {len(corpus_restante)}")

In [ ]:
# 1. CONFIGURAÇÕES VISUAIS E ALTA RESOLUÇÃO
%config InlineBackend.figure_format = 'retina'
sns.set_theme(style="whitegrid")

# 2. PREPARAÇÃO DOS DOIS GRUPOS (Total vs Amostra)
# Criando o DataFrame do Dataset Total Aprovado (Apenas os mantidos)
df_total = pd.DataFrame(corpus)
df_total['Grupo'] = 'Corpus Total (Filtrado)'

# Criando o DataFrame da Amostra de 3000
df_amostra = pd.DataFrame(corpus_amostrado)
df_amostra['Grupo'] = 'Amostra (N=3000)'

# Consolida os dois grupos em um único DataFrame para fins de comparação
df_validacao = pd.concat([df_total, df_amostra], ignore_index=True)

# 3. ENGENHARIA DE RECURSOS UNIFICADA
df_validacao['contagem_palavras'] = df_validacao['body'].apply(lambda x: len(str(x).split()) if x else 0)
df_validacao['tipo_resposta'] = df_validacao.apply(
    lambda r: 'Direta ao Post' if r['parent_id'] == r['link_id'] else 'Thread/Comentário', axis=1
)

# Tratamento de datas e fuso horário de Brasília
df_validacao['datetime'] = (pd.to_datetime(df_validacao['created_utc'], unit='s')
                            .dt.tz_localize('UTC')
                            .dt.tz_convert('America/Sao_Paulo')
                            .dt.tz_localize(None))
df_validacao['hora'] = df_validacao['datetime'].dt.hour
df_validacao['ano_mes'] = df_validacao['datetime'].dt.to_period('M').astype(str)
df_validacao['dia_semana'] = df_validacao['datetime'].dt.day_name().map({
    'Monday': 'Segunda', 'Tuesday': 'Terça', 'Wednesday': 'Quarta',
    'Thursday': 'Quinta', 'Friday': 'Sexta', 'Saturday': 'Sábado', 'Sunday': 'Domingo'
})
ordem_dias = ['Segunda', 'Terça', 'Quarta', 'Quinta', 'Sexta', 'Sábado', 'Domingo']


# 4. CRIAÇÃO DO DASHBOARD DE VALIDAÇÃO EXPANDIDO (3 linhas x 2 colunas)
fig, axes = plt.subplots(3, 2, figsize=(16, 18))

# --- [0, 0] G1: Representatividade por Subreddit ---
sns.histplot(data=df_validacao, x='subreddit', hue='Grupo', multiple='dodge', shrink=0.7,
             stat='percent', common_norm=False, palette='Set1', ax=axes[0, 0])
axes[0, 0].set_title('1. Equilíbrio de Fontes (Subreddit)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Subreddit')
axes[0, 0].set_ylabel('Porcentagem dentro do Grupo (%)')

# --- [0, 1] G2: Distribuição do Tamanho dos Textos (KDE) ---
sns.kdeplot(data=df_validacao, x='contagem_palavras', hue='Grupo', fill=True,
            common_norm=False, palette='Set1', alpha=0.2, ax=axes[0, 1])
if not df_validacao.empty:
    axes[0, 1].set_xlim(0, df_validacao['contagem_palavras'].quantile(0.95))
axes[0, 1].set_title('2. Densidade do Comprimento dos Textos', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Número de Palavras')
axes[0, 1].set_ylabel('Densidade Probabilística')

# --- [1, 0] G3: Estrutura do Fluxo Conversacional ---
sns.histplot(data=df_validacao, x='tipo_resposta', hue='Grupo', multiple='dodge', shrink=0.7,
             stat='percent', common_norm=False, palette='Set1', ax=axes[1, 0])
axes[1, 0].set_title('3. Tipo de Resposta (Estrutura da Árvore)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Tipo de Resposta')
axes[1, 0].set_ylabel('Porcentagem dentro do Grupo (%)')

# --- [1, 1] G4: Evolução Temporal Proporcional (Ano-Mês) ---
df_mes_prop = (df_validacao.groupby(['Grupo', 'ano_mes']).size() / df_validacao.groupby('Grupo').size()).reset_index(name='proporcao')
df_mes_prop['proporcao'] *= 100
sns.lineplot(data=df_mes_prop, x='ano_mes', y='proporcao', hue='Grupo', hue_order=['Corpus Total (Filtrado)', 'Amostra (N=3000)'], marker='o', palette='Set1', linewidth=2, ax=axes[1, 1])
axes[1, 1].set_title('4. Evolução Histórica (Proporção Mensal)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Período (Ano-Mês)')
axes[1, 1].set_ylabel('Porcentagem do Volume Total (%)')
axes[1, 1].tick_params(axis='x', rotation=45)

# --- [2, 0] G5: Ritmo de Atividade por Hora do Dia ---
df_hora_prop = (df_validacao.groupby(['Grupo', 'hora']).size() / df_validacao.groupby('Grupo').size()).reset_index(name='proporcao')
df_hora_prop['proporcao'] *= 100
sns.lineplot(data=df_hora_prop, x='hora', y='proporcao', hue='Grupo', hue_order=['Corpus Total (Filtrado)', 'Amostra (N=3000)'], marker='o', palette='Set1', linewidth=2, ax=axes[2, 0])
axes[2, 0].set_title('5. Volume por Hora do Dia (Horário de Brasília)', fontsize=12, fontweight='bold')
axes[2, 0].set_xlabel('Hora do Dia (0h - 23h)')
axes[2, 0].set_ylabel('Porcentagem do Volume Total (%)')
axes[2, 0].set_xticks(range(0, 24, 2))

# --- [2, 1] G6: Distribuição Proporcional por Dia da Semana ---
df_dia_prop = (df_validacao.groupby(['Grupo', 'dia_semana']).size() / df_validacao.groupby('Grupo').size()).reset_index(name='proporcao')
df_dia_prop['proporcao'] *= 100
sns.barplot(data=df_dia_prop, x='dia_semana', y='proporcao', hue='Grupo', hue_order=['Corpus Total (Filtrado)', 'Amostra (N=3000)'], order=ordem_dias, palette='Set1', ax=axes[2, 1])
axes[2, 1].set_title('6. Volume por Dia da Semana', fontsize=12, fontweight='bold')
axes[2, 1].set_xlabel('Dia da Semana')
axes[2, 1].set_ylabel('Porcentagem do Volume Total (%)')

# Ajustes automáticos de margem para evitar sobreposição
plt.tight_layout()
plt.show()
plt.close('all')

### 1.4 - Anotação Humana

#### 1.4.1 - Divisão do Corpus Não-Anotado

In [ ]:
with open("corpus_amostrado.json", "r", encoding="utf-8") as f:
      corpus_amostrado = json.load(f)

df = pd.DataFrame(corpus_amostrado)
df.head()

In [ ]:
df = df.drop(columns=["id", "name", "author", "subreddit_id", "link_id", "parent_id", "created_utc", "retrieved_on", "score", "ups", "downs",
                 "controversiality", "gilded", "edited", "archived", "score_hidden", "permalink", "filtered"])
df = df.rename(columns={"body": "comentario"})
df["rotulo"] = None
df.head()

In [ ]:
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)

dfs = []

# O skf.split calcula os índices ideais baseando-se na coluna 'subreddit'
for _, indices_fatia in skf.split(df, df['subreddit']):
    # Extrai os 500 registros desta fatia específica
    fatia = df.iloc[indices_fatia].copy()

    # Embaralha os dados internamente para o anotador não receber os subreddits em bloco
    fatia = fatia.sample(frac=1, random_state=42).reset_index(drop=True)

    dfs.append(fatia)

for df_anotador in dfs:
    print(df_anotador['subreddit'].value_counts())

In [ ]:
import openpyxl
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.styles import Font, PatternFill, Alignment

nomes = ["renato", "edvandro", "guilherme", "henrique", "karl", "maicon"]

# Guia de Anotação
guia_texto = """GUIA DE ANOTAÇÃO - PROJETO PLN

Instruções para os Anotadores:
Por favor, classifique cada comentário na coluna 'rótulo' utilizando apenas as opções do menu suspenso: B, N ou M.

Critérios de Classificação:

B (Bom / Suporte):
- Definição: Acolhe, encoraja, oferece ajuda, demonstra empatia ativa.
- Exemplo: "Estou aqui por você, vai ficar tudo bem. Se precisar conversar, pode me chamar."

N (Neutro):
- Definição: Informa, pergunta, relata fatos ou experiências sem uma carga emocional ou suporte explícito.
- Exemplo: "Isso é comum em casos de ansiedade. Você já tentou procurar um profissional?"

M (Mau / Não suporte):
- Definição: Minimiza o sofrimento, demonstra ironia hostil, faz piada depreciativa ou ofensa direta.
- Exemplo: "Para de drama, todo mundo tem problema. Isso é falta de lote pra capinar."
"""

# Geração de arquivos
for fatia, nome in zip(dfs, nomes):
    # Inicializa um arquivo Excel em memória
    wb = openpyxl.Workbook()

    # Aba 1: Guia de Anotação
    ws_guia = wb.active
    ws_guia.title = "Guia de Anotação"
    ws_guia.views.sheetView[0].showGridLines = True

    # Escrever o texto do guia linha por linha
    for r_idx, linha in enumerate(guia_texto.split('\n'), start=1):
        ws_guia.cell(row=r_idx, column=1, value=linha)
    ws_guia.cell(row=1, column=1).font = Font(bold=True, size=14)
    ws_guia.column_dimensions['A'].width = 110

    # Aba 2: Dados do Anotador (com o nome específico)
    ws_dados = wb.create_sheet(title=nome)
    ws_dados.views.sheetView[0].showGridLines = True

    # Escrever nas planilhas os headers solicitados
    headers = ["comentário", "rótulo"]
    ws_dados.append(headers)

    # Formatação visual do cabeçalho (Negrito e Fundo Cinza Claro)
    for col_idx in range(1, 3):
        cell = ws_dados.cell(row=1, column=col_idx)
        cell.font = Font(bold=True)
        cell.fill = PatternFill(start_color="F2F2F2", end_color="F2F2F2", fill_type="solid")

    # Preencher as linhas de dados
    for idx, row in fatia.iterrows():
        linha_atual = ws_dados.max_row + 1

        # Adiciona os valores as celulas de comentário e rótulo
        celula_comentario = ws_dados.cell(row=linha_atual, column=1, value=row["comentario"])
        celula_rotulo = ws_dados.cell(row=linha_atual, column=2, value="")

        # wrap_text=True força a quebra de linha vertical
        celula_comentario.alignment = Alignment(wrap_text=True, vertical="top")
        celula_rotulo.alignment = Alignment(vertical="top")

    # Definir larguras de colunas ideais para leitura humana no Sheets
    ws_dados.column_dimensions['A'].width = 90  # Coluna 'comentário' (antiga B)
    ws_dados.column_dimensions['B'].width = 12  # Coluna 'rótulo' (antiga C)

    # Criar a Regra de Validação de Dados (Menu Suspenso Restrito)
    dv = DataValidation(type="list", formula1='"B,N,M"', allow_blank=True)
    dv.error = 'Entrada inválida! Escolha estritamente entre as opções B, N ou M.'
    dv.errorTitle = 'Erro de Digitação'
    dv.prompt = 'Selecione uma categoria válida para o comentário.'
    dv.promptTitle = 'Classificação'

    # Injetar a validação na planilha de dados
    ws_dados.add_data_validation(dv)

    # Aplica validação
    total_linhas = len(fatia) + 1
    dv.add(f"B2:B{total_linhas}")

    # Salvar o arquivo formatado
    nome_arquivo = f"planilha_{nome}.xlsx"
    wb.save(nome_arquivo)

    print(f"Arquivo {nome_arquivo} gerado com sucesso.")

#### 1.4.2 - União do Corpus Anotado

In [ ]:
import gdown
import json
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

In [ ]:
FILE_IDS = ["1Yjyk3bevf8iSfUyfWdfWSaDc879plBnoThC3A2JxNmA",
            "1bB8a7XXJs1YJWiWcQ-lvsHzg3UxkOHXlMVltT6AVuYI",
            "1r_5AKO1gylC6XPGGiWzCG39X0RJJvCqxgterdTBHiPQ",
            "17ZhuZ2tnKh736_Gf_mEWpZJF5yh1zywag1yjTLENaEU",
            "1RKvEn2VtrbImXgbA52mfFpCLRRIZ2R_oxqU-LDXgzi4",
            "1GiyjMZHOEKon2EphHe7JCGqa_89UYCjivE7NHQ25Ilw"
            ]

nomes = ["edvandro", "guilherme", "henrique", "karl", "maicon", "renato"]

# planilha_edvandro.xlsx - https://docs.google.com/spreadsheets/d/1Yjyk3bevf8iSfUyfWdfWSaDc879plBnoThC3A2JxNmA/edit?usp=drive_link
# planilha_guilherme.xlsx - https://docs.google.com/spreadsheets/d/1bB8a7XXJs1YJWiWcQ-lvsHzg3UxkOHXlMVltT6AVuYI/edit?usp=drive_link
# planilha_henrique.xlsx - https://docs.google.com/spreadsheets/d/1r_5AKO1gylC6XPGGiWzCG39X0RJJvCqxgterdTBHiPQ/edit?usp=sharing
# planilha_karl.xlsx - https://docs.google.com/spreadsheets/d/17ZhuZ2tnKh736_Gf_mEWpZJF5yh1zywag1yjTLENaEU/edit?usp=sharing
# planilha_maicon.xlsx - https://docs.google.com/spreadsheets/d/1RKvEn2VtrbImXgbA52mfFpCLRRIZ2R_oxqU-LDXgzi4/edit?usp=sharing
# planilha_renato.xlsx - https://docs.google.com/spreadsheets/d/1GiyjMZHOEKon2EphHe7JCGqa_89UYCjivE7NHQ25Ilw/edit?usp=sharing

for nome, file_id in zip(nomes, FILE_IDS):
    gdown.download(f'https://drive.google.com/uc?id={file_id}', f'planilha_anotada_{nome}.xlsx', quiet=False)

In [ ]:
corpus_anotado = []

# Baixa os arquivos
for nome in nomes:
    nome_arquivo = f"planilha_anotada_{nome}.xlsx"
    nome_aba = nome

    df_anotador = pd.read_excel(nome_arquivo, sheet_name=nome_aba)

    # df_anotador['anotador'] = nome # Salva o nome do anotador (descomenta se considerar útil)

    # Guarda o DataFrame na lista
    corpus_anotado.append(df_anotador)

# Junta todas as planilhas em um único DataFrame final
df = pd.concat(corpus_anotado, ignore_index=True)

df.rename(columns={'comentário': 'comentario', 'rótulo': 'rotulo'}, inplace=True)
df.dropna(subset=['rotulo'], inplace=True)

df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. PREPARAÇÃO DA MÉTRICA DE TAMANHO
# Conta o número de palavras para relacionarmos com o rótulo
df['contagem_palavras'] = df['comentario'].apply(lambda x: len(str(x).split()))

# 2. CONFIGURAÇÕES VISUAIS
sns.set_theme(style="whitegrid")
# Definindo cores semânticas: B (Verde/Suporte), N (Cinza/Neutro), M (Vermelho/Mau)
cores_rotulos = {'B': '#2ecc71', 'N': '#95a5a6', 'M': '#e74c3c'}
ordem_rotulos = ['B', 'N', 'M']

# 3. CRIAÇÃO DO DASHBOARD (2 Linhas x 2 Colunas)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dashboard de Validação das Anotações Humanas', fontsize=16, fontweight='bold')

# --- [0, 0] G1: Gráfico de Barras (Contagem Absoluta) ---
sns.countplot(data=df, x='rotulo', order=ordem_rotulos, palette=cores_rotulos, ax=axes[0, 0])
axes[0, 0].set_title('1. Volumetria por Rótulo', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Classificação (B=Bom, N=Neutro, M=Mau)')
axes[0, 0].set_ylabel('Quantidade de Comentários')

# Adiciona o número exato em cima de cada barra
for container in axes[0, 0].containers:
    axes[0, 0].bar_label(container, fmt='%d', padding=3)

# --- [0, 1] G2: Gráfico de Pizza (Proporção Relativa) ---
contagem = df['rotulo'].value_counts()
labels_presentes = [r for r in ordem_rotulos if r in contagem.index]
valores = [contagem[r] for r in labels_presentes]
cores_pizza = [cores_rotulos[r] for r in labels_presentes]

axes[0, 1].pie(valores, labels=labels_presentes, autopct='%1.1f%%', startangle=90,
               colors=cores_pizza, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0, 1].set_title('2. Proporção do Dataset', fontsize=12, fontweight='bold')

# --- [1, 0] G3: Boxplot do Tamanho do Texto por Rótulo ---
sns.boxplot(data=df, x='rotulo', y='contagem_palavras', order=ordem_rotulos,
            palette=cores_rotulos, ax=axes[1, 0], showfliers=False)
axes[1, 0].set_title('3. Comprimento do Comentário por Rótulo (Sem Outliers)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Classificação')
axes[1, 0].set_ylabel('Número de Palavras')

# --- [1, 1] G4: Densidade (KDE) do Tamanho do Texto ---
sns.kdeplot(data=df, x='contagem_palavras', hue='rotulo', hue_order=ordem_rotulos,
            palette=cores_rotulos, fill=True, alpha=0.3, ax=axes[1, 1], common_norm=False)
axes[1, 1].set_title('4. Distribuição de Densidade do Comprimento', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Número de Palavras')
axes[1, 1].set_ylabel('Densidade')

# Limita o eixo X para os 95% menores textos (evita que textos gigantes espremam o gráfico)
if not df.empty:
    axes[1, 1].set_xlim(0, df['contagem_palavras'].quantile(0.95))

# Ajusta espaçamentos
plt.tight_layout()
plt.show()

# 4. LIMPEZA DE MEMÓRIA
plt.close('all')

## 1.5 - Divisão Treino, Validação e Teste

#### 1.5.1 - Método I: Holdout

In [ ]:
TEST_SIZE = 0.1 # 10% pra teste
VAL_SIZE = 0.1 / (1-TEST_SIZE) # 10% pra validação

data_splits_holdout = {}

df_restante, data_splits_holdout['teste'] = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df['rotulo'],
    random_state=42
)

data_splits_holdout['treino'], data_splits_holdout['val'] = train_test_split(
    df_restante,
    test_size=VAL_SIZE,
    stratify=df_restante['rotulo'],
    random_state=42
)

print(f"Dados totais: {len(df)}")
print(f"Teste: {len(data_splits_holdout['teste'])}")
print(f"Validação: {len(data_splits_holdout['val'])}")
print(f"Treino: {len(data_splits_holdout['treino'])}")

#### 1.5.2 - Método 2: K-Fold

In [ ]:
TEST_SIZE = 0.1 # 10% pra teste
K_FOLDS = 5

data_splits_k_fold = {}

df_restante, data_splits_k_fold['teste'] = train_test_split(
    df,
    test_size=TEST_SIZE,
    stratify=df['rotulo'],
    random_state=42
)

df_restante = df_restante.reset_index(drop=True) # Reseta o index pro k-fold

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

X = df_restante['comentario']
y = df_restante['rotulo']

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    # Armazena os DataFrames de treino e validação deste fold específico
    data_splits_k_fold[f'fold_{fold}'] = {
        'treino': df_restante.iloc[train_idx].copy(),
        'validacao': df_restante.iloc[val_idx].copy()
    }

In [ ]:
print(f"K1 - Treino: {len(data_splits_k_fold['fold_1']['treino'])} | Validação: {len(data_splits_k_fold['fold_1']['validacao'])}")
print(f"K2 - Treino: {len(data_splits_k_fold['fold_2']['treino'])} | Validação: {len(data_splits_k_fold['fold_2']['validacao'])}")
print(f"K3 - Treino: {len(data_splits_k_fold['fold_3']['treino'])} | Validação: {len(data_splits_k_fold['fold_3']['validacao'])}")
print(f"K4 - Treino: {len(data_splits_k_fold['fold_4']['treino'])} | Validação: {len(data_splits_k_fold['fold_4']['validacao'])}")
print(f"K5 - Treino: {len(data_splits_k_fold['fold_5']['treino'])} | Validação: {len(data_splits_k_fold['fold_5']['validacao'])}")

In [ ]:
data_splits_k_fold['fold_5']['treino']['rotulo'].value_counts()

Nota:
- Em tese, o segundo método é superior, porém consideravelmente mais lento;
- O sentimento do texto costuma estar concentrado nas pontas, então corte fora o meio se precisar;
- Pode rodar a partir do ponto 1.4.2 (+a célula dos imports lá no começo), o resto foi utilizado na criação das planilhas.


## Parte 2 - Modelo

In [ ]:
# Bibliotecas nao-padrao no Colab usadas na Parte 2 (modelo).
# transformers precisa ser recente (>=4.46): o codigo usa eval_strategy e processing_class.
#
# IMPORTANTE: NAO atualizamos 'datasets' de proposito. No Colab, 'pip install -U datasets'
# puxa um pyarrow novo que conflita com o ja carregado pelo pandas (Parte 1) e quebra o
# 'from datasets import Dataset' com 'ValueError: pyarrow.lib.IpcReadOptions size changed'.
# O 'datasets'/'accelerate' que ja vem no Colab bastam para este notebook.
!pip install -q -U transformers accelerate

### 2.1 - Fine-tuning do BERTimbau

Ajuste fino do **BERTimbau** (`neuralmind/bert-base-portuguese-cased`) para classificar os
comentários em 3 classes: **B** (Bom), **N** (Neutro) e **M** (Mau).

Usamos a divisão **holdout** (`data_splits_holdout`) criada na seção 1.5.1. Como a classe **M**
é minoritária, aplicamos **pesos de classe** na função de perda e acompanhamos o **F1 macro**
(que dá peso igual a todas as classes) além da acurácia.

#### Etapa 2.1.1 — Configuração e bibliotecas

Carregamos as bibliotecas e definimos os parâmetros do experimento:

- **`MODEL_NAME`** — o BERTimbau base (`neuralmind/bert-base-portuguese-cased`), um BERT pré-treinado em português.
- **`MAX_LEN = 128`** — todo comentário é truncado em 128 *tokens*; textos maiores são cortados. Cobre a grande maioria dos comentários e mantém o treino rápido.
- **`label2id` / `id2label`** — o modelo trabalha com números, então mapeamos os rótulos `B`, `N`, `M` para `0`, `1`, `2` (e o caminho inverso, para interpretar as previsões).
- **`device`** — usa a **GPU** (`cuda`) se disponível; caso contrário, CPU.

In [ ]:
import os
os.environ.pop("SSLKEYLOGFILE", None)  # evita PermissionError do ssl ao importar transformers

import numpy as np
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# --- Configurações ---
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"  # BERTimbau base
MAX_LEN    = 128
ORDEM_ROTULOS = ['B', 'N', 'M']
label2id = {rot: i for i, rot in enumerate(ORDEM_ROTULOS)}
id2label = {i: rot for rot, i in label2id.items()}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Precisao de treino conforme o hardware:
# - bf16 exige Ampere+ (A100/L4, compute capability >= 8). Colab T4 (7.5) NAO suporta -> usa fp16.
USAR_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
USAR_FP16 = torch.cuda.is_available() and not USAR_BF16
print("Precisao:", "bf16" if USAR_BF16 else ("fp16" if USAR_FP16 else "fp32 (CPU)"))


#### Etapa 2.1.2 — (Opcional) TAPT: pré-treino adaptativo à tarefa

Antes do fine-tuning, aplicamos o **TAPT** (*Task-Adaptive Pre-Training*, Gururangan et al., 2020): continuamos o pré-treino **MLM** (*Masked Language Modeling*) do BERTimbau sobre os **~10 mil comentários NÃO rotulados** (`corpus_restante.json`) — a parte do corpus que não passou pela anotação. Sem usar rótulos, o encoder se adapta ao **vocabulário e ao tom do domínio** (desabafos / ansiedade e depressão no Reddit) antes de aprender a classificar.

- **Como funciona** — mascaramos 15% dos tokens a cada batch e o modelo aprende a prevê-los; o encoder adaptado vira o ponto de partida do classificador (a cabeça de classificação é nova, treinada no fine-tuning).
- **Integração** — ao final, `MODEL_NAME` passa a apontar para `./bertimbau_tapt`, então **Holdout (2.1) e K-Fold (2.2) já partem do checkpoint adaptado**. Para desligar, `USAR_TAPT = False`.
- **Efeito** — o modelo prevê melhor as palavras do domínio (perplexidade menor) e o resultado final fica **mais estável entre as partições** e um pouco mais preciso no agregado. A classe minoritária não se beneficia — isso é desbalanceamento, não domínio, e fica a cargo da focal loss.

In [ ]:
# ==========================================================================
# TAPT - Task-Adaptive Pre-Training (pre-treino adaptativo a tarefa)
# Continua o MLM do BERTimbau nos ~10k comentarios NAO rotulados antes do
# fine-tuning supervisionado (Gururangan et al., 2020).
# ==========================================================================
import os, json, math, gc
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForMaskedLM, set_seed,
                          DataCollatorForLanguageModeling, Trainer, TrainingArguments)

USAR_TAPT  = True                                       # False -> pula o TAPT (usa o BERTimbau base)
MODEL_BASE = "neuralmind/bert-base-portuguese-cased"    # ponto de partida do MLM (sempre o base)
TAPT_DIR   = "./bertimbau_tapt"                          # onde o encoder adaptado sera salvo

def _modelo_salvo(d):
    """True se 'd' contem um modelo HuggingFace ja salvo (nao assume a pasta - verifica)."""
    return os.path.isdir(d) and os.path.exists(os.path.join(d, "config.json"))

if not USAR_TAPT:
    print("TAPT desativado (USAR_TAPT=False) - usando o BERTimbau base.")
elif _modelo_salvo(TAPT_DIR):
    # Encoder adaptado ja existe em disco -> reusa, sem refazer o MLM (que e caro).
    MODEL_NAME = TAPT_DIR
    print(f"TAPT ja existe em {TAPT_DIR} - reusando (sem refazer o pre-treino).")
    print(f"OK - fine-tuning agora parte de: {MODEL_NAME}")
else:
    # 1) Comentarios nao rotulados (mesmo campo 'body' ja normalizado da tarefa)
    with open("corpus_restante.json", "r", encoding="utf-8") as f:
        corpus_nr = json.load(f)
    textos_tapt = [c["body"] for c in corpus_nr
                   if c.get("body") and len(c["body"].strip()) >= 3]
    print(f"Comentarios nao rotulados para TAPT: {len(textos_tapt)}")

    # 2) Tokenizacao (mesma janela MAX_LEN do classificador) + split p/ perplexidade
    tok_tapt = AutoTokenizer.from_pretrained(MODEL_BASE)
    ds_tapt = Dataset.from_dict({"text": textos_tapt})
    ds_tapt = ds_tapt.map(lambda b: tok_tapt(b["text"], truncation=True, max_length=MAX_LEN),
                          batched=True, remove_columns=["text"])
    ds_tapt = ds_tapt.train_test_split(test_size=0.05, seed=42)

    # MLM dinamico: mascara 15% dos tokens a cada batch
    collator_mlm = DataCollatorForLanguageModeling(
        tokenizer=tok_tapt, mlm=True, mlm_probability=0.15)

    # 3) Modelo de MLM a partir do BERTimbau base
    set_seed(42)
    mlm_model = AutoModelForMaskedLM.from_pretrained(MODEL_BASE)

    args_tapt = TrainingArguments(
        output_dir="./tapt_ckpt",
        eval_strategy="epoch",
        save_strategy="no",
        num_train_epochs=3,
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        report_to="none",
        bf16=USAR_BF16,
        fp16=USAR_FP16,
        seed=42,
    )

    trainer_tapt = Trainer(
        model=mlm_model, args=args_tapt,
        train_dataset=ds_tapt["train"], eval_dataset=ds_tapt["test"],
        data_collator=collator_mlm,
    )

    # Perplexidade ANTES (BERTimbau base) e DEPOIS (adaptado) -> evidencia da adaptacao
    ppl_antes = math.exp(trainer_tapt.evaluate()["eval_loss"])
    print(f"Perplexidade (base, antes do TAPT): {ppl_antes:.2f}")

    trainer_tapt.train()

    ppl_depois = math.exp(trainer_tapt.evaluate()["eval_loss"])
    print(f"Perplexidade (apos TAPT):           {ppl_depois:.2f}")

    # 4) Salva o encoder adaptado + tokenizer
    mlm_model.save_pretrained(TAPT_DIR)
    tok_tapt.save_pretrained(TAPT_DIR)

    # 5) Reaponta MODEL_NAME -> as celulas de fine-tuning (Holdout 2.1 e K-Fold 2.2)
    #    usam from_pretrained(MODEL_NAME), logo ja partem do checkpoint adaptado.
    MODEL_NAME = TAPT_DIR
    print()
    print(f"OK - fine-tuning agora parte de: {MODEL_NAME}")

    del mlm_model, trainer_tapt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


#### Etapa 2.1.3 — Tokenização dos dados

O BERT não recebe texto cru: cada comentário é quebrado em *tokens* e convertido em IDs numéricos.

- **`AutoTokenizer`** carrega o tokenizador do próprio BERTimbau (mesmo vocabulário usado no pré-treino).
- **`preparar_dataset`** converte cada *split* (`treino`, `val`, `teste`) em um `Dataset` da Hugging Face, cria a coluna `labels` (o rótulo já em número) e aplica a tokenização com `truncation`/`max_length`.
- **`DataCollatorWithPadding`** monta cada *batch* aplicando *padding* dinâmico — preenche as frases até o tamanho da maior frase do *batch*, em vez de sempre até 128, o que economiza memória e tempo.

In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preparar_dataset(df_split):
    """Converte um split (DataFrame com 'comentario'/'rotulo') em um HF Dataset tokenizado."""
    d = df_split.copy()
    d = d.dropna(subset=['comentario', 'rotulo'])
    d['labels'] = d['rotulo'].map(label2id)
    ds = Dataset.from_pandas(d[['comentario', 'labels']], preserve_index=False)
    ds = ds.map(lambda b: tokenizer(b['comentario'], truncation=True, max_length=MAX_LEN),
                batched=True)
    return ds

train_ds = preparar_dataset(data_splits_holdout['treino'])
val_ds   = preparar_dataset(data_splits_holdout['val'])
test_ds  = preparar_dataset(data_splits_holdout['teste'])

print(f"Treino: {len(train_ds)} | Validação: {len(val_ds)} | Teste: {len(test_ds)}")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Etapa 2.1.4 — Modelo, perda (Focal Loss) e configuração do treino

Aqui montamos tudo que o treinador precisa:

- **Pesos de classe** — como a classe **M** é minoritária (~12%), usamos `compute_class_weight('balanced', ...)`. Eles entram como o **α** da focal loss.
- **`AutoModelForSequenceClassification`** — carrega o BERTimbau e adiciona uma **cabeça de classificação** nova (não treinada) com 3 saídas. O aviso "MISSING" ao carregar é esperado: é essa cabeça que será treinada.
- **`compute_metrics`** — acurácia e **F1** (macro e ponderado). O **F1 macro** trata as 3 classes com a mesma importância.
- **`FocalTrainer` (Focal Loss)** — em vez de só ponderar a perda por classe, a focal loss $-(1-p_t)^{\gamma}\log(p_t)$ **foca nos exemplos difíceis**: exemplos fáceis ($p_t$ alto) quase não contribuem. Isso dá melhor equilíbrio precisão/recall na minoria do que o peso estático sozinho (que tendia a **super-prever M**). Usamos $\gamma=2$ e os pesos balanceados como $\alpha$.
- **`TrainingArguments`** — *learning rate* `2e-5`, **2 épocas** (a partir da ép. 2 o modelo já overfittava), *batch* 16, *warmup*, e `load_best_model_at_end` pelo **F1 macro** na validação.
- **`set_seed(42)`** — fixa a inicialização da cabeça de classificação; sem isso, runs idênticos oscilavam ~0,05 no F1-macro só por causa da init aleatória.

> A focal loss foi validada em um **A/B pareado** (mesma inicialização, 5 folds): ganho médio de **+0,007** no F1-macro, vencendo em **4/5 folds** — pequeno, porém consistente.

---

**Por que a focal loss (estado → consequência → referência):**
- **Estado que motivou:** com a entropia cruzada ponderada por classe, a matriz de confusão mostrava o modelo **super-prevendo M** — 52 predições para 36 reais, precisão de apenas 0,31. O peso estático (M=2,74) corrigia o desbalanceamento *em excesso*, empurrando a fronteira de decisão e transformando muitos **N em M**.
- **Como ataca:** a focal loss pondera por **dificuldade do exemplo**, não por classe inteira — concentra o gradiente nos casos ambíguos (fronteira N↔M) em vez de inflar todos os M.
- **Consequência (medida):** A/B pareado (mesma inicialização, 5 folds) → **+0,007 de F1-macro, vencendo em 4/5 folds**, com melhor recall da minoria. Efeito colateral: probabilidades mais suaves — parte da sub-confiança tratada na Etapa 6.
- **Referências:** Lin et al. (2017), *Focal Loss for Dense Object Detection*, ICCV; Mukhoti et al. (2020), *Calibrating Deep Neural Networks using Focal Loss*, NeurIPS.

In [ ]:
from transformers import set_seed
set_seed(42)   # init determinística da cabeça de classificação -> resultados reprodutíveis

# Pesos de classe (balanced) -> entram como alpha da focal loss; compensam a classe minoritária (M)
y_treino = data_splits_holdout['treino']['rotulo'].map(label2id).values
pesos = compute_class_weight('balanced', classes=np.arange(len(ORDEM_ROTULOS)), y=y_treino)
class_weights = torch.tensor(pesos, dtype=torch.float)
print("Pesos por classe (alpha)", dict(zip(ORDEM_ROTULOS, pesos.round(3))))

# Modelo BERTimbau com cabeça de classificação para 3 classes
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(ORDEM_ROTULOS), id2label=id2label, label2id=label2id)

# Métricas: acurácia + F1 (macro e ponderado)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy":    accuracy_score(labels, preds),
        "f1_macro":    f1_score(labels, preds, average="macro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
    }

# Trainer com FOCAL LOSS: -(1-p_t)^gamma * log(p_t), com peso de classe (alpha).
# Foca nos exemplos difíceis em vez de inflar uma classe inteira (como fazia o peso estático).
class FocalTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logp = torch.nn.functional.log_softmax(outputs.logits, dim=-1)
        ce   = torch.nn.functional.nll_loss(
            logp, labels, weight=self.class_weights.to(logp.device), reduction="none")
        p_t  = logp.gather(1, labels.unsqueeze(1)).squeeze(1).exp()   # prob. da classe correta
        loss = ((1 - p_t) ** self.gamma * ce).mean()                 # fator focal
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./resultados_bertimbau",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=20,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=1,
    report_to="none",
    bf16=USAR_BF16,  # RTX (Blackwell) suporta bf16
    fp16=USAR_FP16,
    seed=42,
)

trainer = FocalTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    gamma=2.0,
)

#### Etapa 2.1.5 — Treinamento (fine-tuning)

`trainer.train()` executa o ajuste fino: percorre os dados de treino pelas 3 épocas, atualizando os pesos do modelo, e ao final de cada época avalia no conjunto de **validação**. Ao terminar, mantém automaticamente os pesos da época com o melhor **F1 macro**.

> Em GPU leva poucos minutos; em CPU é consideravelmente mais lento.

In [ ]:
# Treinamento (fine-tuning). Em GPU leva poucos minutos; em CPU e bem mais lento.
import os
if os.path.isdir("./bertimbau_finetuned") and os.path.exists("./bertimbau_finetuned/config.json"):
    # Modelo holdout ja treinado em disco -> recarrega no trainer e pula o treino.
    print("Modelo holdout ja existe em ./bertimbau_finetuned - recarregando (sem treinar).")
    trainer.model = AutoModelForSequenceClassification.from_pretrained(
        "./bertimbau_finetuned").to(device)
else:
    trainer.train()


#### Etapa 2.1.6 — Avaliação no conjunto de teste

Por fim, medimos o desempenho no conjunto de **teste** — dados que o modelo nunca viu no treino nem na validação, dando uma estimativa honesta da generalização:

- **`classification_report`** — precisão, *recall* e F1 por classe (B, N, M).
- **Matriz de confusão** — mostra onde o modelo acerta e confunde cada classe.
- **`save_model`** — salva o modelo ajustado em `./bertimbau_finetuned` para reuso posterior.

> *Nota:* testamos **calibração de limiares** (otimizar a decisão na validação), mas ela
> **overfitou** o conjunto de validação (~300 exemplos, só ~36 de M) e **piorou o F1-macro no teste**.
> Por isso mantemos o `argmax` puro como decisão final.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Avaliação final no conjunto de TESTE (não usado no treino/validação)
pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)   # decisão = argmax (sem calibração)

# Guarda as métricas do holdout para comparar com o k-fold mais adiante
metricas_holdout = {
    "accuracy":    accuracy_score(y_true, y_pred),
    "f1_macro":    f1_score(y_true, y_pred, average="macro", zero_division=0),
    "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
}

print("== Métricas no Teste ==")
print({k: round(v, 4) for k, v in metricas_holdout.items()})
print()
print(classification_report(y_true, y_pred, target_names=ORDEM_ROTULOS, zero_division=0))

# Matriz de confusão
cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(ORDEM_ROTULOS)))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=ORDEM_ROTULOS, yticklabels=ORDEM_ROTULOS)
plt.title('Matriz de Confusão - BERTimbau (Teste)')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

# Salva o modelo ajustado
trainer.save_model("./bertimbau_finetuned")
tokenizer.save_pretrained("./bertimbau_finetuned")
print("Modelo salvo em ./bertimbau_finetuned")

#### Etapa 2.1.7 — Calibração da confiança (Temperature Scaling)

O modelo é **sub-confiante** (a confiança máxima mal passa de 0,83 e nunca chega a 0,9), embora
acerte bastante quando confia. O **temperature scaling** corrige isso: divide os *logits* por um
escalar $T$ aprendido na **validação** (minimizando a NLL) antes do softmax.

- $T < 1$ **afia** as probabilidades (mais confiante); $T > 1$ **suaviza**.
- É um transform **monotônico** → o `argmax` não muda → **acurácia/F1 idênticos**. Conserta só a confiança.
- **1 único parâmetro** → robusto mesmo na validação pequena (~300), ao contrário da calibração por classe que overfitou.
- Medimos a melhora pelo **ECE** (Expected Calibration Error) e pelo **diagrama de confiabilidade** antes/depois.

---

**Por que temperature scaling (estado → consequência → referência):**
- **Estado que motivou:** o modelo estava **sistematicamente sub-confiante** — confiança máxima ~0,83 (nunca chegava a 0,9), mediana ~0,53 e **ECE = 0,092**. Quando confiava (>0,7) acertava 93–100%, mas raramente confiava (a própria focal loss suaviza as probabilidades).
- **Por que este método (e não calibração por classe):** o $T$ tem **1 único parâmetro** → robusto na validação pequena (~300). A calibração por classe (3 parâmetros) testada antes **overfitou** e piorou o F1-macro no teste.
- **Consequência (medida):** $T \approx 0,65$ (afia); **ECE 0,092 → 0,023 (−75%)**, confiança máxima 0,83 → 0,94, **sem alterar acurácia/F1** (transform monotônico).
- **Referência:** Guo et al. (2017), *On Calibration of Modern Neural Networks*, ICML.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# --- Ajusta T na VALIDAÇÃO (minimiza NLL) e aplica no TESTE ---
logits_val  = trainer.predict(val_ds).predictions
logits_test = trainer.predict(test_ds).predictions
y_val   = data_splits_holdout['val']['rotulo'].map(label2id).values
y_teste = data_splits_holdout['teste']['rotulo'].map(label2id).values

_lv = torch.tensor(logits_val, dtype=torch.float); _yv = torch.tensor(y_val)
def _nll(T):
    return torch.nn.functional.cross_entropy(_lv / T, _yv).item()
T = minimize_scalar(_nll, bounds=(0.05, 10.0), method="bounded").x
print(f"Temperatura ótima T = {T:.3f}  "
      f"({'afia: o modelo era SUB-confiante' if T < 1 else 'suaviza: o modelo era SUPER-confiante'})")

def _softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

def _ece(probs, labels, n_bins=10):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum(): e += abs(conf[m].mean() - acc[m].mean()) * m.sum()
    return e / len(conf)

probs_antes  = _softmax(logits_test)
probs_depois = _softmax(logits_test / T)

from sklearn.metrics import accuracy_score, f1_score
print(f"\nAcurácia:  antes={accuracy_score(y_teste, probs_antes.argmax(1)):.4f}  "
      f"depois={accuracy_score(y_teste, probs_depois.argmax(1)):.4f}  (T não muda o argmax)")
print(f"f1_macro:  antes={f1_score(y_teste, probs_antes.argmax(1), average='macro', zero_division=0):.4f}  "
      f"depois={f1_score(y_teste, probs_depois.argmax(1), average='macro', zero_division=0):.4f}")
print(f"Confiança média:  {probs_antes.max(1).mean():.3f} -> {probs_depois.max(1).mean():.3f}")
print(f"Confiança máxima: {probs_antes.max(1).max():.3f} -> {probs_depois.max(1).max():.3f}")
print(f"%% predições >0.9: {(probs_antes.max(1)>0.9).mean()*100:.1f}%% -> {(probs_depois.max(1)>0.9).mean()*100:.1f}%%")
print(f"ECE (erro de calibração): {_ece(probs_antes, y_teste):.4f} -> {_ece(probs_depois, y_teste):.4f}")

# Diagrama de confiabilidade antes vs depois
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, probs, titulo in [(axes[0], probs_antes, "Antes (T=1)"),
                          (axes[1], probs_depois, f"Depois (T={T:.2f})")]:
    conf = probs.max(1); acc = (probs.argmax(1) == y_teste).astype(float)
    bins = np.linspace(0, 1, 11); xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum(): xs.append(conf[m].mean()); ys.append(acc[m].mean())
    ax.plot([0, 1], [0, 1], '--', color='gray', label='calibração perfeita')
    ax.plot(xs, ys, 'o-', color='#4A90D9')
    ax.set_title(titulo); ax.set_xlabel('Confiança'); ax.set_ylabel('Acurácia')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend()
plt.suptitle('Diagrama de Confiabilidade — Temperature Scaling', fontweight='bold')
plt.tight_layout(); plt.show()

# Salva a temperatura junto do modelo (para reuso na inferência)
import json as _json
with open("./bertimbau_finetuned/temperatura.json", "w", encoding="utf-8") as f:
    _json.dump({"temperatura": float(T)}, f)
print(f"\nTemperatura T={T:.3f} salva em ./bertimbau_finetuned/temperatura.json")

### 2.2 - Validação Cruzada (K-Fold)

Em vez de confiar em uma única divisão treino/validação (holdout), a **validação cruzada k-fold**
treina e avalia o modelo **K vezes**, cada vez usando um *fold* diferente como validação. Assim
obtemos uma estimativa mais **robusta** do desempenho (média ± desvio entre os folds), menos
sensível à "sorte" de uma divisão específica.

Reutilizamos os mesmos *folds* criados na seção 1.5.2 (`data_splits_k_fold`) e as funções já
definidas na seção 2.1 (`preparar_dataset`, `compute_metrics`, `FocalTrainer`, `tokenizer`).
A contrapartida é o **custo**: treinamos o BERTimbau K vezes, então leva ~K× mais tempo que o holdout.

---

**Por que validação cruzada (estado → consequência → referência):**
- **Estado que motivou:** um único *holdout* tem alta variância — o F1-macro oscilava ~0,05 entre execuções idênticas. Com um split só não dá para saber se uma mudança ajudou ou foi sorte.
- **Consequência:** a média ± desvio entre os 5 folds dá uma estimativa **mais robusta e com incerteza quantificada**; foi o que permitiu validar a focal loss (A/B pareado) e flagrar o overfitting a partir da 2ª época.
- **Referência:** Kohavi (1995), *A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection*, IJCAI.

#### Etapa 2.2.1 — Treino e avaliação por fold

Para cada um dos `K` folds:

1. Separamos o `treino` e a `validacao` daquele fold.
2. Recalculamos os **pesos de classe** com base apenas no treino daquele fold.
3. Criamos um **modelo novo** (`from_pretrained`) — é essencial reiniciar os pesos a cada fold, senão haveria vazamento de aprendizado entre eles.
4. Treinamos e avaliamos na `validacao` do fold, guardando as métricas.
5. Liberamos a memória da GPU (`del` + `empty_cache`) antes do próximo fold.

In [ ]:
import gc
from transformers import set_seed

K_FOLDS = 5
NUM_EPOCAS_KF = 2   # 2 epocas: a partir da ep. 2 o val loss subia (overfitting) e o F1 ja estava no plato
resultados_folds = []

# Para o MODELO FINAL (ensemble): salvamos cada modelo do fold e guardamos os logits no TESTE,
# alem das predicoes out-of-fold (validacao de cada fold) para calibrar o T do ensemble.
import os
os.makedirs("./modelos_kfold", exist_ok=True)
test_ds_kf = preparar_dataset(data_splits_k_fold['teste'])
y_teste_kf = data_splits_k_fold['teste']['rotulo'].map(label2id).values
logits_folds_teste = []   # logits de cada fold no TESTE
logits_oof = []           # logits out-of-fold (validacao de cada fold)
y_oof = []                # rotulos correspondentes

for fold in range(1, K_FOLDS + 1):
    print(f"===== Fold {fold}/{K_FOLDS} =====")
    treino_fold = data_splits_k_fold[f'fold_{fold}']['treino']
    val_fold    = data_splits_k_fold[f'fold_{fold}']['validacao']

    # Datasets tokenizados deste fold
    train_ds_kf = preparar_dataset(treino_fold)
    val_ds_kf   = preparar_dataset(val_fold)

    # Pesos de classe balanceados a partir do treino DESTE fold
    y_kf = treino_fold['rotulo'].map(label2id).values
    pesos_kf = compute_class_weight('balanced', classes=np.arange(len(ORDEM_ROTULOS)), y=y_kf)
    cw_kf = torch.tensor(pesos_kf, dtype=torch.float)

    # Nao assume a pasta: verifica se o modelo deste fold ja existe em disco.
    fold_dir = f"./modelos_kfold/fold_{fold}"
    ja_treinado = os.path.isdir(fold_dir) and os.path.exists(os.path.join(fold_dir, "config.json"))

    set_seed(42 + fold)   # init deterministica por fold (relevante no treino)
    if ja_treinado:
        print(f"Fold {fold}: modelo ja existe em {fold_dir} - recarregando (sem treinar).")
        model_kf = AutoModelForSequenceClassification.from_pretrained(fold_dir)
    else:
        # Modelo novo (pesos reiniciados a cada fold)
        model_kf = AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME, num_labels=len(ORDEM_ROTULOS), id2label=id2label, label2id=label2id)

    args_kf = TrainingArguments(
        output_dir=f"./resultados_kfold/fold_{fold}",
        eval_strategy="no",
        save_strategy="no",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=NUM_EPOCAS_KF,
        weight_decay=0.01,
        warmup_ratio=0.1,
        logging_steps=50,
        report_to="none",
        bf16=USAR_BF16,
        fp16=USAR_FP16,
        seed=42,
    )

    trainer_kf = FocalTrainer(
        model=model_kf, args=args_kf,
        train_dataset=train_ds_kf, eval_dataset=val_ds_kf,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics, class_weights=cw_kf,
    )

    # Treina e salva so se ainda nao havia modelo deste fold em disco
    if not ja_treinado:
        trainer_kf.train()
        trainer_kf.save_model(fold_dir)

    m = trainer_kf.evaluate()
    print(f"Fold {fold}: acc={m['eval_accuracy']:.4f} | "
          f"f1_macro={m['eval_f1_macro']:.4f} | f1_weighted={m['eval_f1_weighted']:.4f}")

    resultados_folds.append({
        'fold': fold,
        'accuracy':    m['eval_accuracy'],
        'f1_macro':    m['eval_f1_macro'],
        'f1_weighted': m['eval_f1_weighted'],
    })

    # Logits no TESTE e out-of-fold (para o ensemble + calibracao da secao 2.5)
    logits_folds_teste.append(trainer_kf.predict(test_ds_kf).predictions)
    logits_oof.append(trainer_kf.predict(val_ds_kf).predictions)
    y_oof.append(val_fold['rotulo'].map(label2id).values)

    # Libera memoria da GPU antes do proximo fold
    del model_kf, trainer_kf
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

tokenizer.save_pretrained("./modelos_kfold")
print("\n5 modelos do k-fold prontos em ./modelos_kfold")


#### Etapa 2.2.2 — Agregação dos resultados

Consolidamos as métricas dos `K` folds e calculamos **média ± desvio padrão**. A média indica o
desempenho esperado do modelo; o desvio mostra o quanto ele varia conforme a divisão dos dados
(quanto menor, mais estável). Comparando com o resultado do holdout (seção 2.1), temos uma leitura
mais confiável da qualidade do fine-tuning.

In [ ]:
import matplotlib.pyplot as plt

df_kf = pd.DataFrame(resultados_folds).set_index('fold')
print("Métricas por fold:")
print(df_kf.round(4))

print("Média ± desvio padrão entre os folds:")
for col in ['accuracy', 'f1_macro', 'f1_weighted']:
    print(f"  {col:12}: {df_kf[col].mean():.4f} ± {df_kf[col].std():.4f}")

# Gráfico de barras: desempenho por fold
ax = df_kf[['accuracy', 'f1_macro']].plot(kind='bar', figsize=(8, 4))
ax.set_title('Desempenho por Fold (Validação Cruzada)')
ax.set_xlabel('Fold')
ax.set_ylabel('Métrica')
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### 2.3 - Comparação: Holdout vs K-Fold

Reunimos em uma única tabela o resultado do **holdout** (seção 2.1, medido no conjunto de teste) e a
**média dos folds** da validação cruzada (seção 2.2). O desvio padrão do k-fold indica a estabilidade
do modelo entre diferentes divisões dos dados.

> Observação: o holdout é medido no **teste** e o k-fold na **validação** de cada fold — não são exatamente
> o mesmo conjunto, mas a comparação dá uma boa noção de consistência do desempenho.

In [ ]:
# Junta os resultados das duas estratégias numa única tabela.
# (Requer ter rodado a seção 2.1 -> metricas_holdout e a seção 2.2 -> df_kf)
METRICAS = ['accuracy', 'f1_macro', 'f1_weighted']

comparacao = pd.DataFrame({
    'Holdout (teste)':   pd.Series(metricas_holdout),
    'K-Fold (média)':    df_kf[METRICAS].mean(),
    'K-Fold (desvio)':   df_kf[METRICAS].std(),
}).loc[METRICAS].round(4)
comparacao.index.name = 'métrica'

print("== Comparação Holdout x K-Fold ==")
print(comparacao)

# Gráfico de barras: Holdout vs média do K-Fold (com barra de erro = desvio dos folds)
import matplotlib.pyplot as plt
import numpy as np

x = np.arange(len(METRICAS))
largura = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - largura/2, comparacao['Holdout (teste)'], largura, label='Holdout (teste)')
ax.bar(x + largura/2, comparacao['K-Fold (média)'],  largura,
       yerr=comparacao['K-Fold (desvio)'], capsize=4, label='K-Fold (média ± desvio)')
ax.set_xticks(x)
ax.set_xticklabels(METRICAS)
ax.set_ylim(0, 1)
ax.set_ylabel('Métrica')
ax.set_title('BERTimbau: Holdout vs Validação Cruzada')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

comparacao

### 2.4 - Ensemble dos folds (média dos logits)

Em vez de descartar os 5 modelos do k-fold, fazemos um **deep ensemble**: cada fold prevê no
**mesmo conjunto de teste** e nós **mediamos os logits** dos 5 → uma única predição (a média de logits permite
aplicar o temperature scaling depois, na seção 2.5).

- **Memória:** os modelos foram usados **um de cada vez** dentro do loop (cél. 54) — só guardamos
  os vetores de logits (minúsculos). Não há 5 modelos na memória ao mesmo tempo.
- **Por quê:** o ensemble costuma dar predições **mais estáveis e melhor calibradas** que um único
  modelo, e a discordância entre os membros é, por si, um sinal de incerteza. Comparamos abaixo o
  **modelo único (holdout)** vs **a média dos 5 folds (individual)** vs **o ensemble**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

def _softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

# Ensemble = softmax da MÉDIA DOS LOGITS dos 5 folds (logits permitem aplicar o T na seção 2.5)
probs_ensemble = _softmax(np.mean(logits_folds_teste, axis=0))
y_pred_ens = probs_ensemble.argmax(1)

def _met(yp):
    return {"accuracy":    accuracy_score(y_teste_kf, yp),
            "f1_macro":    f1_score(y_teste_kf, yp, average="macro", zero_division=0),
            "f1_weighted": f1_score(y_teste_kf, yp, average="weighted", zero_division=0)}

# Desempenho de cada fold individualmente no teste
met_individuais = [_met(L.argmax(1)) for L in logits_folds_teste]
print("f1_macro de cada fold no teste:", [round(m["f1_macro"], 4) for m in met_individuais])

metricas_ensemble = _met(y_pred_ens)

comparacao_ens = pd.DataFrame({
    "Modelo único (holdout)":         metricas_holdout,
    "Média dos 5 folds (individual)": {k: np.mean([m[k] for m in met_individuais]) for k in metricas_holdout},
    "Ensemble (média dos logits)":    metricas_ensemble,
}).T.round(4)

print("\n== Comparação no conjunto de TESTE ==")
print(comparacao_ens)
print()
print(classification_report(y_teste_kf, y_pred_ens, target_names=ORDEM_ROTULOS, zero_division=0))

cm = confusion_matrix(y_teste_kf, y_pred_ens, labels=np.arange(len(ORDEM_ROTULOS)))
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=ORDEM_ROTULOS, yticklabels=ORDEM_ROTULOS)
plt.title('Matriz de Confusão - Ensemble dos 5 folds (Teste, sem calibração)')
plt.xlabel('Predito')
plt.ylabel('Real')
plt.tight_layout()
plt.show()

### 2.5 - Modelo Final: Ensemble + Temperature Scaling

Aqui consolidamos **o modelo usado em todo o resto do notebook** (Parte 3 e Parte 4):

1. **Ensemble** dos 5 modelos do k-fold (treinados com focal loss) — média dos *logits*.
2. **Temperature scaling** — o $T$ é ajustado nas predições **out-of-fold** (a validação de cada
   fold, que aquele modelo não viu → sem vazamento) e aplicado sobre os logits médios.

A função **`prever_ensemble(textos)`** encapsula tudo: carrega os 5 modelos (um de cada vez, para
não estourar memória), faz a média dos logits, aplica o $T$ e devolve probabilidades calibradas na
ordem `[B, N, M]`. A partir daqui, **toda predição do BERTimbau** (corpus_restante na Seção 3 e
conjunto de teste na Parte 4) passa por essa função — então o ensemble e a calibração se refletem
em **todos os gráficos e métricas seguintes**.

---

**Por que ensemble + temperature scaling como modelo final (estado → consequência → referência):**
- **Estado que motivou:** mesmo com `set_seed`, um modelo único tem variância; e a confiança seguia baixa (~0,55) até no ensemble cru. Queríamos um modelo final **mais estável e bem calibrado**, reaproveitando os 5 modelos que o k-fold já treina (memória controlada: 1 modelo por vez).
- **Consequência (medida):** o ensemble supera a média dos folds individuais (**+0,003 F1**) e é mais estável; com o $T$ ajustado *out-of-fold*, o **ECE cai ~57%** sem mudar a acurácia. A discordância entre os 5 membros é, por si, um sinal de incerteza.
- **Referência:** Lakshminarayanan et al. (2017), *Simple and Scalable Predictive Uncertainty Estimation using Deep Ensembles*, NeurIPS.

In [ ]:
import os, gc, json
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.optimize import minimize_scalar

device = "cuda" if torch.cuda.is_available() else "cpu"
MODELOS_ENSEMBLE = [f"./modelos_kfold/fold_{k}" for k in range(1, K_FOLDS + 1)]
tok_final = AutoTokenizer.from_pretrained("./modelos_kfold")

# --- Temperatura do ensemble: ajustada nas predições OUT-OF-FOLD (sem vazamento) ---
_lo = torch.tensor(np.vstack(logits_oof), dtype=torch.float)
_yo = torch.tensor(np.concatenate(y_oof))
T_ENSEMBLE = float(minimize_scalar(
    lambda T: torch.nn.functional.cross_entropy(_lo / T, _yo).item(),
    bounds=(0.05, 10.0), method="bounded").x)
print(f"T do ensemble (ajustado out-of-fold) = {T_ENSEMBLE:.3f}")

def _softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True)); return e / e.sum(axis=1, keepdims=True)

def prever_ensemble(textos, batch=64):
    """MODELO FINAL: média dos logits dos 5 folds + temperature scaling.
    Carrega 1 modelo por vez (memória) e devolve probabilidades calibradas na ordem [B, N, M]."""
    soma = None
    for path in MODELOS_ENSEMBLE:
        mdl = AutoModelForSequenceClassification.from_pretrained(path).to(device).eval()
        L = []
        for i in range(0, len(textos), batch):
            enc = tok_final(textos[i:i+batch], truncation=True, max_length=128,
                            padding=True, return_tensors="pt").to(device)
            with torch.no_grad():
                L.append(mdl(**enc).logits.cpu().numpy())
        L = np.vstack(L)
        soma = L if soma is None else soma + L
        del mdl; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return _softmax(soma / len(MODELOS_ENSEMBLE) / T_ENSEMBLE)

# Persiste a configuração do modelo final
with open("./modelos_kfold/modelo_final.json", "w", encoding="utf-8") as f:
    json.dump({"modelos": MODELOS_ENSEMBLE, "temperatura": T_ENSEMBLE, "ordem": ORDEM_ROTULOS}, f)
print("Modelo final:", len(MODELOS_ENSEMBLE), "folds + T =", round(T_ENSEMBLE, 3))

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt

def _ece(probs, labels, n_bins=10):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum(): e += abs(conf[m].mean() - acc[m].mean()) * m.sum()
    return e / len(conf)

# Modelo final no TESTE: ensemble sem vs com calibração
logit_ens = np.mean(logits_folds_teste, axis=0)
probs_raw = _softmax(logit_ens)
probs_cal = _softmax(logit_ens / T_ENSEMBLE)

print("== Modelo final (ensemble) no conjunto de TESTE ==")
print(f"f1_macro = {f1_score(y_teste_kf, probs_cal.argmax(1), average='macro', zero_division=0):.4f}"
      f"  |  accuracy = {accuracy_score(y_teste_kf, probs_cal.argmax(1)):.4f}  (o T não muda o argmax)")
print(f"Confiança média: {probs_raw.max(1).mean():.3f} (sem T) -> {probs_cal.max(1).mean():.3f} (com T)")
print(f"ECE:             {_ece(probs_raw, y_teste_kf):.4f} (sem T) -> {_ece(probs_cal, y_teste_kf):.4f} (com T)")
print()
print(classification_report(y_teste_kf, probs_cal.argmax(1), target_names=ORDEM_ROTULOS, zero_division=0))

# Diagrama de confiabilidade do modelo final (com T)
conf = probs_cal.max(1); acc = (probs_cal.argmax(1) == y_teste_kf).astype(float)
bins = np.linspace(0, 1, 11); xs, ys = [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (conf >= lo) & (conf < hi)
    if m.sum(): xs.append(conf[m].mean()); ys.append(acc[m].mean())
plt.figure(figsize=(5, 4))
plt.plot([0, 1], [0, 1], '--', color='gray', label='calibração perfeita')
plt.plot(xs, ys, 'o-', color='#2e8b57')
plt.title(f'Confiabilidade - Modelo Final (Ensemble + T={T_ENSEMBLE:.2f})')
plt.xlabel('Confiança'); plt.ylabel('Acurácia'); plt.xlim(0, 1); plt.ylim(0, 1); plt.legend()
plt.tight_layout(); plt.show()

## Parte 3 - Avaliação

A construção do Dataset foi realizada pelos próprios membros do grupo, o que implica em não haver rótulos que possam ser usados como ground truth.

Idealmente o Dataset deveria ser rotulado por um especialista na área da saúde, preferencialmente que trabalhasse com saúde mental. Contudo, não foi possível utilizar essa opção. Então os membros do grupo fizeram os rótulos para análise de sentimentos de três mil comentários.

Devido ao escopo do trabalho, da proposta inicial e da quantidade de tempo, foram deixados de lado aspectos como concordância entre os anotadores e Cohen's kappa ou kappa de Fleiss, sendo realizada uma única anotação de cada comentário.

Devido a isso, nos limitamos a utilizar o dataset anotado somente para treinar o modelo, pois a avaliação não seria adequada dado que não é possível garantir a validade da anotação.

Portanto, a avaliação se deu através de uma comparação. O modelo que treinamos foi comparado a um modelo do estado da arte de classificação de sentimentos, que este sim, foi treinado e testado com um ground truth verificado e validado acadêmicamente.

Portanto, a avaliação aqui abaixo visa analisar o quão próximo o modelo que treinamos fica de um modelo do estado da arte e não o quão perto nosso modelo fica perto do ground truth que fizemos.

Após uma revisão bibliográfica, não encontrou-se modelos de grande porte e refência que se focavam no Reddit específicamente. No entanto, muito foi encontrado sobre o avanço utilizando específicamente o Twitter. Os comentários utilizados nesse trabalho foram principalmente curtos, que são bem próximos aos encontrados no Twitter. Devido a compatibilidade de contexto, optamos por utilizar modelos de referência utilizados no Twitter.

Os modelos considerados no estado da arte foram:

**[Pysentimiento](https://huggingface.co/pysentimiento/bertweet-pt-sentiment)**: Este modelo possui versões pré-treinadas com textos em espanhol e português. A versão em português é baseada no ajuste fino utilizando o modelo BERTabaporu [[Pablo Bottonda Costa, 2022]](https://doi.org/10.26615/978-954-452-092-2_024) em um corpus de treinamento de 15.000 tweets em português brasileiro [[Brum e das Graças Volpe Nunes, 2018]](
https://doi.org/10.48550/arXiv.1712.08917
), anotados manualmente para a tarefa de análise de sentimentos.

Referência:

[pysentimiento: A Python Toolkit for Opinion Mining and Social NLP tasks](https://doi.org/10.48550/arXiv.2106.09462)

**[XLM-RoBERTa](https://huggingface.co/cardiffnlp/twitter-xlm-roberta-base-sentiment)**: Baseado no modelo BERT, a versão do [XLM-RoBERTa](https://doi.org/10.48550/arXiv.1911.02116) (Cross Lingual Language Model - Robustly Optimized BERT-Pretraining Approach) utilizada neste trabalho foi previamente treinada em aproximadamente 10 milhões de tweets em português, especificamente para a tarefa de classificação de sentimentos.

Referência:

[XLM-T: Multilingual Language Models in Twitter for Sentiment Analysis and Beyond](
https://doi.org/10.48550/arXiv.2104.12250)

Referências gerais:

[Sentiment Analysis of Shared Content in Brazilian RedditCommunities](https://doi.org/10.5753/jis.2025.5564)

[Sentiment analysis in Portuguese tweets: an evaluation
of diverse word representation models](https://doi.org/10.1007/s10579-023-09661-4)

**Seção 3.5 — Análise Geral**

*Análise Individual dos Modelos*
- A distribuição de classes (POS/NEU/NEG) de cada um dos 4 modelos é apresentada em gráficos de barras individuais.
- A evolução temporal da distribuição de classes ao longo dos meses é exibida como lineplot para os 3 modelos neurais.
- A mediana de confiança por classe é calculada e comparada entre os 3 modelos neurais em um gráfico de barras agrupado.
- Histogramas de distribuição de confiança por classe são apresentados para cada modelo neural, com a mediana indicada.
- A taxa de predições com confiança alta (>0.9) versus incerta (<0.6) por classe é exibida para cada modelo neural.

*Análise Comparativa Geral dos 4 Modelos*
- O percentual de amostras onde todos os modelos concordam (4/4), onde 3 concordam, onde 2 concordam e onde nenhum concorda é apresentado em gráfico de barras.
- Um heatmap exibe a taxa de concordância entre todos os pares de modelos.
- Exemplos representativos de cada nível de concordância são exibidos, ordenados por maior certeza agregada dos modelos.

---

**Seção 3.6 — Pysentimiento × BERTimbau**

*Etapa 1 — Comparação de igual para igual*
- Uma matriz de concordância 3×3 entre os dois modelos é apresentada.
- A taxa de concordância total e por classe é calculada e exibida.
- As discordâncias são classificadas em leves (classes adjacentes: POS↔NEU ou NEU↔NEG) e severas (classes opostas: POS↔NEG), com suas respectivas proporções.

*Etapa 2 — Pysentimiento como ground truth*
- Accuracy, Precision, Recall e F1 por classe, F1-macro e F1-weighted do BERTimbau são reportados tendo o Pysentimiento como referência.
- A matriz de confusão é exibida em contagens absolutas e normalizada por linha.
- O Recall da classe NEG recebe destaque com comentário contextualizado ao domínio de saúde mental.

---

**Seção 3.7 — XLM-RoBERTa × BERTimbau**

A mesma metodologia da seção anterior é aplicada, substituindo Pysentimiento por XLM-RoBERTa: matriz de concordância 3×3, taxa de concordância total e por classe, proporção de discordâncias leves versus severas, métricas formais com XLM-RoBERTa como ground truth, matriz de confusão absoluta e normalizada, e destaque do Recall NEG contextualizado ao domínio de saúde mental.

---

**Seção 3.8 — Simbólico × BERTimbau**

- Uma matriz de concordância 3×3 entre o modelo simbólico e o BERTimbau é apresentada.
- A taxa de concordância total e por classe é calculada e exibida.
- A proporção de discordâncias leves versus severas é calculada e apresentada em gráfico de barras.

---

**Seção 3.9 — Ensemble Pysentimiento + XLM-RoBERTa como Ground Truth**

- O Cohen's Kappa entre Pysentimiento e XLM-RoBERTa é calculado sobre o dataset completo, servindo como justificativa da qualidade do ensemble antes de seu uso como referência.
- O dataset é filtrado mantendo apenas os casos onde os dois modelos concordam, com registro dos casos mantidos e descartados e da distribuição de classes no subconjunto resultante, apresentados em gráfico de barras.
- Accuracy, Precision, Recall e F1 por classe, F1-macro e F1-weighted do BERTimbau são reportados contra esse ground truth consensual.
- A matriz de confusão do BERTimbau usando o ensemble como referência é exibida, com destaque para o Recall da classe NEG contextualizado ao domínio de saúde mental.

In [ ]:
import gdown

file_id = '17J5yGvanBfelGNog-Xk6VgaXrF11x-t0'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'corpus_restante_enriquecido.json', quiet=False)

file_id = '1ccsWap02Ku4khkA9Wy2_kzXS1Z-RM6NN'
gdown.download(f'https://drive.google.com/uc?id={file_id}', 'corpus_restante_classificado.json', quiet=False)

In [ ]:
# Carregar o JSON
with open("corpus_restante.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

# Lista de novos campos
novos_campos = [
    # Pysentimiento
    "Pysentimiento_Output",
    "Pysentimiento_POS",
    "Pysentimiento_NEU",
    "Pysentimiento_NEG",

    # XLM-RoBERTa
    "XLM-RoBERTa_Output",
    "XLM-RoBERTa_POS",
    "XLM-RoBERTa_NEU",
    "XLM-RoBERTa_NEG",

    # BERTimbau
    "BERTimbau_Output",
    "BERTimbau_POS",
    "BERTimbau_NEU",
    "BERTimbau_NEG"
]

# Inicializar campos
for item in dados:
    for campo in novos_campos:
        if "Output" in campo:
            item[campo] = ""
        else:
            item[campo] = 0.0

# Imprimir os 2 primeiros com linha em branco entre eles
for item in dados[:2]:
    print(json.dumps(item, ensure_ascii=False, indent=2))
    print()

#### 3.1 - Pysentimiento

In [ ]:
# Instalar dependências
!pip install pysentimiento tqdm

import json
import torch
from pysentimiento import create_analyzer
from tqdm import tqdm

# Verificar GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    print("GPU está sendo usada!")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU NÃO está sendo usada, rodando na CPU.")

# Criar o analisador
analyzer = create_analyzer(task="sentiment", lang="pt", device=device)

# Tamanho do batch
batch_size = 1024

# Processamento em batch com barra de progresso
for i in tqdm(range(0, len(dados), batch_size), desc="Processando sentimentos"):
    batch = dados[i:i+batch_size]

    textos = []
    indices_validos = []

    # Preparar textos válidos
    for j, item in enumerate(batch):
        texto = item.get("body", "")

        if texto and isinstance(texto, str):
            textos.append(texto)
            indices_validos.append(j)

    if not textos:
        continue

    # Inferência em batch
    resultados = analyzer.predict(textos)

    # Garantir lista
    if not isinstance(resultados, list):
        resultados = [resultados]

    # Preencher resultados
    for idx, resultado in zip(indices_validos, resultados):
        item = batch[idx]

        item["Pysentimiento_Output"] = resultado.output

        probas = resultado.probas
        item["Pysentimiento_POS"] = probas.get("POS", 0.0)
        item["Pysentimiento_NEU"] = probas.get("NEU", 0.0)
        item["Pysentimiento_NEG"] = probas.get("NEG", 0.0)

# Imprimir os 2 primeiros
for item in dados[:2]:
    print(json.dumps(item, ensure_ascii=False, indent=2))
    print()

In [ ]:
# Limpar memória da GPU antes de recarregar
import gc
import torch

# Deletar o classifier anterior se existir
try:
    del classifier
except:
    pass

gc.collect()
torch.cuda.empty_cache()

print(f"Memória livre: {torch.cuda.mem_get_info()[0] / 1024**3:.2f} GB")

#### 3.2 - XLM-RoBERTa

In [ ]:
import json
import torch
from transformers import pipeline
from tqdm import tqdm

# Verificar GPU
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("GPU está sendo usada!")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU NÃO está sendo usada, rodando na CPU.")

# Criar o pipeline XLM-RoBERTa
model_path = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
classifier = pipeline(
    "sentiment-analysis",
    model=model_path,
    tokenizer=model_path,
    device=device,
    truncation=True,
    max_length=512
)

# Mapeamento dos labels do modelo para POS/NEU/NEG
# Mapeamento dos labels do modelo para POS/NEU/NEG
label_map = {
    "positive": "POS",
    "neutral":  "NEU",
    "negative": "NEG"
}

# Tamanho do batch
batch_size = 512

# Processamento em batch com barra de progresso
for i in tqdm(range(0, len(dados), batch_size), desc="Processando XLM-RoBERTa"):
    batch = dados[i:i+batch_size]

    textos = []
    indices_validos = []

    # Preparar textos válidos
    for j, item in enumerate(batch):
        texto = item.get("body", "")
        if texto and isinstance(texto, str):
            textos.append(texto)
            indices_validos.append(j)

    if not textos:
        continue

    # Inferência em batch
    resultados_top = classifier(
        textos,
        batch_size=batch_size
    )

    resultados_all = classifier(
        textos,
        top_k=None,
        batch_size=batch_size
    )

    # Preencher resultados
    for idx, (top, scores) in zip(indices_validos, zip(resultados_top, resultados_all)):
        item = batch[idx]

        probas = {label_map[s["label"]]: s["score"] for s in scores}

        item["XLM-RoBERTa_Output"] = label_map[top["label"]]
        item["XLM-RoBERTa_POS"] = probas.get("POS", 0.0)
        item["XLM-RoBERTa_NEU"] = probas.get("NEU", 0.0)
        item["XLM-RoBERTa_NEG"] = probas.get("NEG", 0.0)

# Imprimir os 2 primeiros
for item in dados[:2]:
    print(json.dumps(item, ensure_ascii=False, indent=2))
    print()

#### 3.3 - BERTimbau (Nosso Fine-tuning)

In [ ]:
with open("corpus_restante_classificado.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=2)

print("Salvo com sucesso!")

In [ ]:
with open("corpus_restante_classificado.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

print("Arquivo carregado com sucesso!")

# ----------------------------------------------------------------------
# BERTimbau REAL: usa o modelo final treinado na Secao 2
# (ensemble dos 5 folds + temperature scaling, via prever_ensemble).
# Antes isto era um stub com random.random() -> predicoes ALEATORIAS.
# ----------------------------------------------------------------------
assert "prever_ensemble" in globals(), \
    "prever_ensemble nao existe: rode a Secao 2 (Parte 2 - Modelo) antes desta celula."

import gc, torch

# Libera o XLM-RoBERTa da GPU antes de carregar os modelos do BERTimbau
try:
    del classifier
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# prever_ensemble devolve probabilidades na ordem [B, N, M].
# Convencao da Parte 3 e POS/NEU/NEG -> B(Bom)=POS, N(Neutro)=NEU, M(Mau)=NEG.
BNM_PARA_PNN = {"B": "POS", "N": "NEU", "M": "NEG"}

textos = [str(d.get("body") or "") for d in dados]
probs_bert = prever_ensemble(textos)   # shape (len(dados), 3), ordem [B, N, M]

for idx, d in enumerate(dados):
    p = probs_bert[idx]
    d["BERTimbau_POS"] = float(p[0])   # B
    d["BERTimbau_NEU"] = float(p[1])   # N
    d["BERTimbau_NEG"] = float(p[2])   # M
    d["BERTimbau_Output"] = BNM_PARA_PNN[ORDEM_ROTULOS[int(p.argmax())]]

# Regrava o corpus agora com os 3 modelos (Pysentimiento, XLM-RoBERTa, BERTimbau)
with open("corpus_restante_classificado.json", "w", encoding="utf-8") as f:
    json.dump(dados, f, ensure_ascii=False, indent=2)

print("BERTimbau (modelo real da Secao 2) preenchido com sucesso!")

# visualizar 2 exemplos
for d in dados[:2]:
    print(json.dumps(d, ensure_ascii=False, indent=2))


#### 3.4 - Simbólico

In [ ]:
# carregar simbólico
with open("corpus_restante_enriquecido.json", "r", encoding="utf-8") as f:
    dados_simbolico = json.load(f)

# mapa de conversão
mapa = {
    "Positivo": "POS",
    "Negativo": "NEG",
    "Neutro": "NEU"
}

# merge + normalização
for i in range(len(dados)):
    dados[i]["Symbolic_Output"] = mapa.get(dados_simbolico[i]["Symbolic_Output"], "NEU")
    dados[i]["Symbolic_Score"] = dados_simbolico[i]["Symbolic_Score"]
    dados[i]["Symbolic_Terms"] = dados_simbolico[i]["Symbolic_Terms"]

print("Merge com normalização concluído!")

# conferir 2 exemplos
for d in dados[:2]:
    print(json.dumps(d, ensure_ascii=False, indent=2))

#### 3.5 - Análise Geral

In [ ]:
# ============================================================
# SEÇÃO 1 — Análise Individual dos Modelos
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from scipy.stats import entropy as scipy_entropy

df = pd.DataFrame(dados)

# ── Paleta e estilo ─────────────────────────────────────────
PALETTE = {"POS": "#4CAF82", "NEU": "#F5A623", "NEG": "#E05C5C"}
MODELS_NEURAL = ["Pysentimiento", "XLM-RoBERTa", "BERTimbau"]
MODELS_ALL    = ["Pysentimiento", "XLM-RoBERTa", "BERTimbau", "Symbolic"]
CLASS_ORDER   = ["POS", "NEU", "NEG"]

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False,
                     "axes.spines.right": False})

CONF_COLS = {
    "Pysentimiento": {"POS": "Pysentimiento_POS", "NEU": "Pysentimiento_NEU", "NEG": "Pysentimiento_NEG"},
    "XLM-RoBERTa":  {"POS": "XLM-RoBERTa_POS",   "NEU": "XLM-RoBERTa_NEU",  "NEG": "XLM-RoBERTa_NEG"},
    "BERTimbau":    {"POS": "BERTimbau_POS",       "NEU": "BERTimbau_NEU",    "NEG": "BERTimbau_NEG"},
}

df["text_len"] = df["body"].astype(str).str.len()
df["date"]     = pd.to_datetime(df["created_utc"], unit="s")
df["month"]    = df["date"].dt.to_period("M")


# ════════════════════════════════════════════════════════════
# 1.1  Distribuição de classes — visão individual por modelo
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, model in zip(axes, MODELS_ALL):
    col    = f"{model}_Output"
    counts = df[col].value_counts(normalize=True).reindex(CLASS_ORDER, fill_value=0) * 100
    bars   = ax.bar(CLASS_ORDER, counts.values,
                    color=[PALETTE[c] for c in CLASS_ORDER],
                    edgecolor="white", linewidth=0.8, width=0.55)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_ylim(0, counts.max() * 1.25)
    ax.set_xlabel("Classe")
    ax.set_ylabel("Proporção (%)" if model == MODELS_ALL[0] else "")

fig.suptitle("Distribuição de Classes por Modelo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("sec1_01_distribuicao_classes.png")
plt.show()
print("✔  1.1 — Distribuição individual de classes salva.")


# ════════════════════════════════════════════════════════════
# 1.2  Evolução temporal — modelos neurais
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, model in zip(axes, MODELS_NEURAL):
    col = f"{model}_Output"
    tmp = (df.groupby(["month", col])
             .size()
             .reset_index(name="n"))
    tmp["pct"]      = tmp["n"] / tmp.groupby("month")["n"].transform("sum") * 100
    tmp["month_dt"] = tmp["month"].dt.to_timestamp()

    for cls in CLASS_ORDER:
        sub = tmp[tmp[col] == cls].sort_values("month_dt")
        ax.plot(sub["month_dt"], sub["pct"],
                label=cls, color=PALETTE[cls], linewidth=2, marker="o", markersize=3)

    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Proporção (%)" if model == MODELS_NEURAL[0] else "")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(fontsize=8)

fig.suptitle("Evolução Temporal da Distribuição de Classes (modelos neurais)",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("sec1_02_evolucao_temporal.png", bbox_inches="tight")
plt.show()
print("✔  1.2 — Evolução temporal salva.")


# ════════════════════════════════════════════════════════════
# 1.3  Mediana de confiança por classe — comparação entre modelos
# ════════════════════════════════════════════════════════════

medians = {}
for model in MODELS_NEURAL:
    medians[model] = {}
    for cls in CLASS_ORDER:
        mask = df[f"{model}_Output"] == cls
        medians[model][cls] = df.loc[mask, CONF_COLS[model][cls]].median()

medians_df = pd.DataFrame(medians).T  # shape: (3 models × 3 classes)

x     = np.arange(len(CLASS_ORDER))
width = 0.22
fig, ax = plt.subplots(figsize=(9, 5))

for i, model in enumerate(MODELS_NEURAL):
    offset = (i - 1) * width
    bars = ax.bar(x + offset, [medians_df.loc[model, cls] for cls in CLASS_ORDER],
                  width=width, label=model, edgecolor="white", linewidth=0.7, alpha=0.88)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_ORDER, fontsize=12)
ax.set_ylabel("Mediana de Confiança")
ax.set_ylim(0, 1.1)
ax.set_title("Mediana de Confiança por Classe e Modelo", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig("sec1_03_mediana_confianca.png")
plt.show()
print("✔  1.3 — Mediana de confiança salva.")


# ════════════════════════════════════════════════════════════
# 1.4  Histogramas de confiança por classe — modelos neurais
# ════════════════════════════════════════════════════════════

fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True, sharey=True)

for row, model in enumerate(MODELS_NEURAL):
    for col_idx, cls in enumerate(CLASS_ORDER):
        ax       = axes[row][col_idx]
        conf_col = CONF_COLS[model][cls]
        mask     = df[f"{model}_Output"] == cls
        data     = df.loc[mask, conf_col].dropna()

        ax.hist(data, bins=30, color=PALETTE[cls], edgecolor="white", alpha=0.85, density=True)
        ax.axvline(data.median(), color="black", linestyle="--",
                   linewidth=1.2, label=f"mediana={data.median():.2f}")
        ax.legend(fontsize=7)

        # título apenas na primeira linha; label de classe apenas na última linha
        if row == 0:
            ax.set_title(cls, fontsize=11, fontweight="bold", color=PALETTE[cls])
        if row == 2:
            ax.set_xlabel("Confiança")
        # label de modelo apenas na primeira coluna
        if col_idx == 0:
            ax.set_ylabel(f"{model}\nDensidade", fontsize=9)

fig.suptitle("Distribuição de Confiança por Classe (modelos neurais)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sec1_04_histogramas_confianca.png")
plt.show()
print("✔  1.4 — Histogramas de confiança salvos.")


# ════════════════════════════════════════════════════════════
# 1.5  Taxa de confiança alta vs. incerta por modelo
# ════════════════════════════════════════════════════════════

THRESHOLD_HIGH = 0.9
THRESHOLD_LOW  = 0.6

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)

for ax, model in zip(axes, MODELS_NEURAL):
    output_col = f"{model}_Output"
    rates = {"Alta (>0.9)": [], "Incerta (<0.6)": []}

    for cls in CLASS_ORDER:
        mask     = df[output_col] == cls
        conf_col = CONF_COLS[model][cls]
        confs    = df.loc[mask, conf_col].dropna()
        n        = len(confs)
        if n == 0:
            rates["Alta (>0.9)"].append(0)
            rates["Incerta (<0.6)"].append(0)
        else:
            rates["Alta (>0.9)"].append((confs > THRESHOLD_HIGH).sum() / n * 100)
            rates["Incerta (<0.6)"].append((confs < THRESHOLD_LOW).sum() / n * 100)

    x     = np.arange(len(CLASS_ORDER))
    width = 0.32
    b1 = ax.bar(x - width / 2, rates["Alta (>0.9)"],  width=width,
                label="Alta (>0.9)",  color="#4A90D9", edgecolor="white", alpha=0.88)
    b2 = ax.bar(x + width / 2, rates["Incerta (<0.6)"], width=width,
                label="Incerta (<0.6)", color="#B0B0B0", edgecolor="white", alpha=0.88)

    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{bar.get_height():.1f}%", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_ORDER)
    ax.set_title(model, fontsize=12, fontweight="bold")
    ax.set_xlabel("Classe predita")
    ax.set_ylabel("% das amostras" if model == MODELS_NEURAL[0] else "")
    ax.set_ylim(0, 105)
    ax.legend(fontsize=8)

fig.suptitle("Taxa de Predições com Confiança Alta (>0.9) vs. Incerta (<0.6)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sec1_05_confianca_alta_vs_incerta.png")
plt.show()
print("✔  1.5 — Taxa de confiança alta vs. incerta salva.")

In [ ]:
# ============================================================
# SEÇÃO 2 — Análise Comparativa Geral dos 4 Modelos
# ============================================================

from itertools import combinations
from IPython.display import display

OUTPUT_COLS = {m: f"{m}_Output" for m in MODELS_ALL}

# ── Nível de concordância por amostra ───────────────────────
df["_outputs"] = df[[OUTPUT_COLS[m] for m in MODELS_ALL]].values.tolist()
df["_n_unique"] = df["_outputs"].apply(lambda x: len(set(x)))
df["_max_agree"] = df["_outputs"].apply(lambda x: max(x.count(c) for c in set(x)))

def nivel(row):
    ma = row["_max_agree"]
    if ma == 4: return "Todos concordam (4/4)"
    if ma == 3: return "3 concordam (3/4)"
    if ma == 2: return "2 concordam (2/4)"
    return "Nenhum concorda (1/4)"

df["_nivel"] = df.apply(nivel, axis=1)

NIVEL_ORDER = [
    "Todos concordam (4/4)",
    "3 concordam (3/4)",
    "2 concordam (2/4)",
    "Nenhum concorda (1/4)",
]
NIVEL_COLORS = ["#4CAF82", "#8BC34A", "#F5A623", "#E05C5C"]


# ════════════════════════════════════════════════════════════
# 2.1  Percentual de amostras por nível de concordância
# ════════════════════════════════════════════════════════════

nivel_counts = (df["_nivel"]
                .value_counts(normalize=True)
                .reindex(NIVEL_ORDER, fill_value=0) * 100)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(NIVEL_ORDER, nivel_counts.values,
              color=NIVEL_COLORS, edgecolor="white", linewidth=0.8, width=0.55)

for bar, val in zip(bars, nivel_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylabel("% das amostras")
ax.set_ylim(0, nivel_counts.max() * 1.2)
ax.set_title("Nível de Concordância entre os 4 Modelos", fontsize=14, fontweight="bold")
ax.tick_params(axis="x", labelsize=10)

plt.tight_layout()
plt.savefig("sec2_01_concordancia_nivel.png")
plt.show()
print("✔  2.1 — Nível de concordância salvo.")


# ════════════════════════════════════════════════════════════
# 2.2  Heatmap de taxa de concordância entre pares
# ════════════════════════════════════════════════════════════

n_models = len(MODELS_ALL)
agree_matrix = pd.DataFrame(np.nan, index=MODELS_ALL, columns=MODELS_ALL)

for m in MODELS_ALL:
    agree_matrix.loc[m, m] = 1.0

for m1, m2 in combinations(MODELS_ALL, 2):
    rate = (df[OUTPUT_COLS[m1]] == df[OUTPUT_COLS[m2]]).mean()
    agree_matrix.loc[m1, m2] = rate
    agree_matrix.loc[m2, m1] = rate

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.zeros_like(agree_matrix, dtype=bool)  # sem máscara — mostrar tudo

sns.heatmap(agree_matrix.astype(float),
            annot=True, fmt=".2f", cmap="YlGn",
            vmin=0, vmax=1,
            linewidths=0.5, linecolor="white",
            square=True, ax=ax,
            annot_kws={"size": 12, "weight": "bold"})

ax.set_title("Taxa de Concordância entre Pares de Modelos", fontsize=13, fontweight="bold")
ax.tick_params(axis="x", rotation=30, labelsize=10)
ax.tick_params(axis="y", rotation=0,  labelsize=10)

plt.tight_layout()
plt.savefig("sec2_02_heatmap_concordancia.png")
plt.show()
print("✔  2.2 — Heatmap de concordância salvo.")


# ════════════════════════════════════════════════════════════
# 2.3  Exemplos representativos por nível de concordância
# ════════════════════════════════════════════════════════════

LABEL_COLS = [OUTPUT_COLS[m] for m in MODELS_ALL]
SHOW_COLS  = ["body"] + LABEL_COLS

N_EXAMPLES = 3

# ── Confiança do modelo na sua própria predição ─────────────
for model in MODELS_NEURAL:
    col_out = OUTPUT_COLS[model]
    df[f"_conf_{model}"] = df.apply(
        lambda row: row[CONF_COLS[model][row[col_out]]], axis=1
    )

# Simbólico: sempre 1.0 (determinístico, sem ponderação)
df["_conf_Symbolic"] = 1.0

# ── Score de certeza agregado ────────────────────────────────
df["_certeza_total"] = df[[f"_conf_{m}" for m in MODELS_NEURAL]].sum(axis=1) + 1.0

print("\n" + "═" * 70)
print("2.3 — EXEMPLOS REPRESENTATIVOS POR NÍVEL DE CONCORDÂNCIA")
print("     (ordenados por maior certeza agregada)")
print("═" * 70)

for nivel_name in NIVEL_ORDER:
    subset = (df[df["_nivel"] == nivel_name]
              .sort_values("_certeza_total", ascending=False))
    n_avail = len(subset)
    sample  = subset.head(N_EXAMPLES)

    print(f"\n{'─'*70}")
    print(f"  {nivel_name}  (n={n_avail}, {n_avail/len(df)*100:.1f}% do total)")
    print(f"{'─'*70}")

    for i, (_, row) in enumerate(sample.iterrows(), 1):
        body_preview = str(row["body"])[:200] + ("..." if len(str(row["body"])) > 200 else "")

        labels_conf = " | ".join(
            f"{m}: {row[OUTPUT_COLS[m]]} ({row[f'_conf_{m}']:.2f})"
            for m in MODELS_NEURAL
        )
        symbolic = f"Symbolic: {row[OUTPUT_COLS['Symbolic']]}"

        print(f"\n  [{i}] {body_preview}")
        print(f"       → {labels_conf} | {symbolic}")
        print(f"       → certeza total (neurais): {row['_certeza_total'] - 1.0:.3f} + Symbolic=1.0 = {row['_certeza_total']:.3f}")

print("\n" + "═" * 70)

#### 3.6 - Pysentimiento - BERTimbau

In [ ]:
# ============================================================
# SEÇÃO 3 — Pysentimiento × BERTimbau
# Etapa 3.1 — Comparação de igual para igual
# ============================================================

MODEL_A = "Pysentimiento"
MODEL_B = "BERTimbau"

COL_A = OUTPUT_COLS[MODEL_A]
COL_B = OUTPUT_COLS[MODEL_B]

# ── Severidade da discordância ───────────────────────────────
# Leve:   POS↔NEU ou NEU↔NEG (classes adjacentes)
# Severa: POS↔NEG (classes opostas)
SEVERITY = {
    frozenset({"POS", "NEU"}): "leve",
    frozenset({"NEU", "NEG"}): "leve",
    frozenset({"POS", "NEG"}): "severa",
}

def get_severity(a, b):
    if a == b:
        return "concordância"
    return SEVERITY.get(frozenset({a, b}), "leve")

df["_3_severity"] = df.apply(lambda r: get_severity(r[COL_A], r[COL_B]), axis=1)
df_discord = df[df["_3_severity"] != "concordância"].copy()


# ════════════════════════════════════════════════════════════
# 3.1.1  Matriz de concordância 3×3
# ════════════════════════════════════════════════════════════

conf_matrix = pd.crosstab(
    df[COL_A], df[COL_B],
    rownames=[MODEL_A], colnames=[MODEL_B]
).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, linecolor="white",
            square=True, ax=ax,
            annot_kws={"size": 13, "weight": "bold"})

ax.set_title(f"Matriz de Concordância\n{MODEL_A} × {MODEL_B}",
             fontsize=13, fontweight="bold")
ax.set_xlabel(MODEL_B, fontsize=11)
ax.set_ylabel(MODEL_A, fontsize=11)

# destaca a diagonal
for i in range(len(CLASS_ORDER)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                edgecolor="#2E7D32", linewidth=2.5))

plt.tight_layout()
plt.savefig("sec3_01_matriz_concordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 3.1.2  Taxa de concordância total e por classe
# ════════════════════════════════════════════════════════════

total_agree = (df[COL_A] == df[COL_B]).mean() * 100
print(f"\n── Taxa de concordância total: {total_agree:.2f}%\n")
print(f"── Por classe (amostras onde {MODEL_A} predisse a classe):")

for cls in CLASS_ORDER:
    mask      = df[COL_A] == cls
    agree_cls = (df.loc[mask, COL_A] == df.loc[mask, COL_B]).mean() * 100
    n         = mask.sum()
    print(f"   {cls}: {agree_cls:.2f}%  (n={n})")


# ════════════════════════════════════════════════════════════
# 3.1.3  Proporção de discordâncias leves vs. severas
# ════════════════════════════════════════════════════════════

sev_counts = df_discord["_3_severity"].value_counts()
n_leve     = sev_counts.get("leve",   0)
n_severa   = sev_counts.get("severa", 0)
n_discord  = len(df_discord)

print(f"\n── Discordâncias totais: {n_discord} ({n_discord/len(df)*100:.1f}% das amostras)")
print(f"   Leves  (adjacentes): {n_leve}  ({n_leve/n_discord*100:.1f}%)")
print(f"   Severas (POS↔NEG):   {n_severa} ({n_severa/n_discord*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Leve\n(adjacente)", "Severa\n(POS↔NEG)"],
              [n_leve/n_discord*100, n_severa/n_discord*100],
              color=["#F5A623", "#E05C5C"], edgecolor="white", width=0.45)

for bar, val in zip(bars, [n_leve/n_discord*100, n_severa/n_discord*100]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("% das discordâncias")
ax.set_ylim(0, max(n_leve, n_severa)/n_discord*100 * 1.25)
ax.set_title(f"Tipo de Discordância\n{MODEL_A} × {MODEL_B}",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("sec3_03_tipo_discordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 3.1.4  Scatterplot de confiança nos casos de discordância
# ════════════════════════════════════════════════════════════

# confiança de cada modelo na sua própria predição
df_discord["_conf_A"] = df_discord.apply(
    lambda r: r[CONF_COLS[MODEL_A][r[COL_A]]], axis=1)
df_discord["_conf_B"] = df_discord.apply(
    lambda r: r[CONF_COLS[MODEL_B][r[COL_B]]], axis=1)

fig, ax = plt.subplots(figsize=(7, 6))

for sev, color, marker, zorder in [("leve", "#F5A623", "o", 2),
                                     ("severa", "#E05C5C", "X", 3)]:
    sub = df_discord[df_discord["_3_severity"] == sev]
    ax.scatter(sub["_conf_A"], sub["_conf_B"],
               c=color, marker=marker, alpha=0.45, s=35,
               label=f"Discordância {sev} (n={len(sub)})", zorder=zorder)

ax.axline((0, 0), slope=1, color="grey", linestyle="--",
          linewidth=1, label="confiança igual")
ax.set_xlabel(f"Confiança — {MODEL_A}", fontsize=11)
ax.set_ylabel(f"Confiança — {MODEL_B}", fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f"Confiança nos Casos de Discordância\n{MODEL_A} × {MODEL_B}",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("sec3_04_scatter_discordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 3.1.5  Exemplos de discordância severa
# ════════════════════════════════════════════════════════════

df_severa = (df_discord[df_discord["_3_severity"] == "severa"]
             .sort_values(["_conf_A", "_conf_B"], ascending=False))

N_SEVERE = 5
print(f"\n{'═'*70}")
print(f"3.1.5 — EXEMPLOS DE DISCORDÂNCIA SEVERA (POS↔NEG)  —  top {N_SEVERE}")
print(f"        ordenados por maior certeza dos dois modelos")
print(f"{'═'*70}")

for i, (_, row) in enumerate(df_severa.head(N_SEVERE).iterrows(), 1):
    body_preview = str(row["body"])[:250] + ("..." if len(str(row["body"])) > 250 else "")
    print(f"\n  [{i}] {body_preview}")
    print(f"       → {MODEL_A}: {row[COL_A]} ({row['_conf_A']:.2f})  |  "
          f"{MODEL_B}: {row[COL_B]} ({row['_conf_B']:.2f})")

print(f"\n{'═'*70}")

In [ ]:
# ============================================================
# SEÇÃO 3 — Etapa 3.2
# Pysentimiento como ground truth × BERTimbau
# ============================================================

from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_recall_fscore_support)

y_true = df[COL_A]   # Pysentimiento = ground truth
y_pred = df[COL_B]   # BERTimbau     = predito

# ════════════════════════════════════════════════════════════
# 3.2.1  Métricas por classe + macro + weighted
# ════════════════════════════════════════════════════════════

acc = accuracy_score(y_true, y_pred)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=CLASS_ORDER, zero_division=0)

f1_macro    = f1.mean()
_, _, f1_w, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0)

# ── Tabela de métricas ───────────────────────────────────────
metrics_df = pd.DataFrame({
    "Classe":    CLASS_ORDER,
    "Precision": precision,
    "Recall":    recall,
    "F1":        f1,
    "Support":   support.astype(int),
})

print(f"\n{'═'*60}")
print(f"  3.2 — {MODEL_A} como ground truth × {MODEL_B}")
print(f"{'═'*60}")
print(f"\n  Accuracy : {acc:.4f}\n")
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  F1-macro    : {f1_macro:.4f}")
print(f"  F1-weighted : {f1_w:.4f}")

# ── Destaque NEG ─────────────────────────────────────────────
neg_idx    = CLASS_ORDER.index("NEG")
neg_recall = recall[neg_idx]

print(f"\n{'─'*60}")
print(f"  ⚠  Recall NEG = {neg_recall:.4f}")
print(f"""
  No domínio de saúde mental, falhar em identificar um comentário
  negativo (não-suporte, hostil ou minimizador) é especialmente
  custoso: o modelo deixa passar conteúdo que pode expor usuários
  vulneráveis a respostas prejudiciais sem sinalização.
  Um Recall NEG baixo significa que o BERTimbau sistematicamente
  subestima a hostilidade — risco direto em aplicações de moderação
  ou triagem de conteúdo em comunidades como r/desabafos.
""")
print(f"{'─'*60}")


# ════════════════════════════════════════════════════════════
# 3.2.2  Matriz de confusão
# ════════════════════════════════════════════════════════════

cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)

# normalizada por linha (recall por célula)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Absoluta ─────────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[0], annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title("Contagens absolutas", fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"Predito — {MODEL_B}", fontsize=10)
axes[0].set_ylabel(f"Real — {MODEL_A}", fontsize=10)

# destaca diagonal
for i in range(len(CLASS_ORDER)):
    axes[0].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

# ── Normalizada (recall) ─────────────────────────────────────
annot_norm = np.array([[f"{v:.2f}" for v in row] for row in cm_norm])

sns.heatmap(cm_norm, annot=annot_norm, fmt="", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[1], annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title("Normalizada por linha (Recall)", fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"Predito — {MODEL_B}", fontsize=10)
axes[1].set_ylabel(f"Real — {MODEL_A}", fontsize=10)

# destaca linha NEG inteira para chamar atenção
neg_row = CLASS_ORDER.index("NEG")
axes[1].add_patch(plt.Rectangle((0, neg_row), len(CLASS_ORDER), 1,
                                  fill=False, edgecolor="#E05C5C", linewidth=2.5))
# destaca diagonal
for i in range(len(CLASS_ORDER)):
    axes[1].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

fig.suptitle(f"Matriz de Confusão — {MODEL_A} (ground truth) × {MODEL_B}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sec3_02_matriz_confusao.png")
plt.show()

#### 3.7 - XLM-RoBERTa - BERTimbau

In [ ]:
# ============================================================
# SEÇÃO 4 — XLM-RoBERTa × BERTimbau
# Etapa 4.1 — Comparação de igual para igual
# ============================================================

MODEL_A = "XLM-RoBERTa"
MODEL_B = "BERTimbau"

COL_A = OUTPUT_COLS[MODEL_A]
COL_B = OUTPUT_COLS[MODEL_B]

# ── Severidade da discordância ───────────────────────────────
# Leve:   POS↔NEU ou NEU↔NEG (classes adjacentes)
# Severa: POS↔NEG (classes opostas)
SEVERITY = {
    frozenset({"POS", "NEU"}): "leve",
    frozenset({"NEU", "NEG"}): "leve",
    frozenset({"POS", "NEG"}): "severa",
}

def get_severity(a, b):
    if a == b:
        return "concordância"
    return SEVERITY.get(frozenset({a, b}), "leve")

df["_4_severity"] = df.apply(lambda r: get_severity(r[COL_A], r[COL_B]), axis=1)
df_discord = df[df["_4_severity"] != "concordância"].copy()


# ════════════════════════════════════════════════════════════
# 4.1.1  Matriz de concordância 3×3
# ════════════════════════════════════════════════════════════

conf_matrix = pd.crosstab(
    df[COL_A], df[COL_B],
    rownames=[MODEL_A], colnames=[MODEL_B]
).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, linecolor="white",
            square=True, ax=ax,
            annot_kws={"size": 13, "weight": "bold"})

ax.set_title(f"Matriz de Concordância\n{MODEL_A} × {MODEL_B}",
             fontsize=13, fontweight="bold")
ax.set_xlabel(MODEL_B, fontsize=11)
ax.set_ylabel(MODEL_A, fontsize=11)

# destaca a diagonal
for i in range(len(CLASS_ORDER)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                edgecolor="#2E7D32", linewidth=2.5))

plt.tight_layout()
plt.savefig("sec4_01_matriz_concordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 4.1.2  Taxa de concordância total e por classe
# ════════════════════════════════════════════════════════════

total_agree = (df[COL_A] == df[COL_B]).mean() * 100
print(f"\n── Taxa de concordância total: {total_agree:.2f}%\n")
print(f"── Por classe (amostras onde {MODEL_A} predisse a classe):")

for cls in CLASS_ORDER:
    mask      = df[COL_A] == cls
    agree_cls = (df.loc[mask, COL_A] == df.loc[mask, COL_B]).mean() * 100
    n         = mask.sum()
    print(f"   {cls}: {agree_cls:.2f}%  (n={n})")


# ════════════════════════════════════════════════════════════
# 4.1.3  Proporção de discordâncias leves vs. severas
# ════════════════════════════════════════════════════════════

sev_counts = df_discord["_4_severity"].value_counts()
n_leve     = sev_counts.get("leve",   0)
n_severa   = sev_counts.get("severa", 0)
n_discord  = len(df_discord)

print(f"\n── Discordâncias totais: {n_discord} ({n_discord/len(df)*100:.1f}% das amostras)")
print(f"   Leves  (adjacentes): {n_leve}  ({n_leve/n_discord*100:.1f}%)")
print(f"   Severas (POS↔NEG):   {n_severa} ({n_severa/n_discord*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Leve\n(adjacente)", "Severa\n(POS↔NEG)"],
              [n_leve/n_discord*100, n_severa/n_discord*100],
              color=["#F5A623", "#E05C5C"], edgecolor="white", width=0.45)

for bar, val in zip(bars, [n_leve/n_discord*100, n_severa/n_discord*100]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("% das discordâncias")
ax.set_ylim(0, max(n_leve, n_severa)/n_discord*100 * 1.25)
ax.set_title(f"Tipo de Discordância\n{MODEL_A} × {MODEL_B}",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("sec4_03_tipo_discordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 4.1.4  Scatterplot de confiança nos casos de discordância
# ════════════════════════════════════════════════════════════

df_discord["_conf_A"] = df_discord.apply(
    lambda r: r[CONF_COLS[MODEL_A][r[COL_A]]], axis=1)
df_discord["_conf_B"] = df_discord.apply(
    lambda r: r[CONF_COLS[MODEL_B][r[COL_B]]], axis=1)

fig, ax = plt.subplots(figsize=(7, 6))

for sev, color, marker, zorder in [("leve", "#F5A623", "o", 2),
                                     ("severa", "#E05C5C", "X", 3)]:
    sub = df_discord[df_discord["_4_severity"] == sev]
    ax.scatter(sub["_conf_A"], sub["_conf_B"],
               c=color, marker=marker, alpha=0.45, s=35,
               label=f"Discordância {sev} (n={len(sub)})", zorder=zorder)

ax.axline((0, 0), slope=1, color="grey", linestyle="--",
          linewidth=1, label="confiança igual")
ax.set_xlabel(f"Confiança — {MODEL_A}", fontsize=11)
ax.set_ylabel(f"Confiança — {MODEL_B}", fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title(f"Confiança nos Casos de Discordância\n{MODEL_A} × {MODEL_B}",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("sec4_04_scatter_discordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 4.1.5  Exemplos de discordância severa
# ════════════════════════════════════════════════════════════

df_severa = (df_discord[df_discord["_4_severity"] == "severa"]
             .sort_values(["_conf_A", "_conf_B"], ascending=False))

N_SEVERE = 5
print(f"\n{'═'*70}")
print(f"4.1.5 — EXEMPLOS DE DISCORDÂNCIA SEVERA (POS↔NEG)  —  top {N_SEVERE}")
print(f"        ordenados por maior certeza dos dois modelos")
print(f"{'═'*70}")

for i, (_, row) in enumerate(df_severa.head(N_SEVERE).iterrows(), 1):
    body_preview = str(row["body"])[:250] + ("..." if len(str(row["body"])) > 250 else "")
    print(f"\n  [{i}] {body_preview}")
    print(f"       → {MODEL_A}: {row[COL_A]} ({row['_conf_A']:.2f})  |  "
          f"{MODEL_B}: {row[COL_B]} ({row['_conf_B']:.2f})")

print(f"\n{'═'*70}")

In [ ]:
# ============================================================
# SEÇÃO 4 — Etapa 4.2
# XLM-RoBERTa como ground truth × BERTimbau
# ============================================================

from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_recall_fscore_support)

y_true = df[COL_A]   # XLM-RoBERTa = ground truth
y_pred = df[COL_B]   # BERTimbau   = predito

# ════════════════════════════════════════════════════════════
# 4.2.1  Métricas por classe + macro + weighted
# ════════════════════════════════════════════════════════════

acc = accuracy_score(y_true, y_pred)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=CLASS_ORDER, zero_division=0)

f1_macro    = f1.mean()
_, _, f1_w, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0)

# ── Tabela de métricas ───────────────────────────────────────
metrics_df = pd.DataFrame({
    "Classe":    CLASS_ORDER,
    "Precision": precision,
    "Recall":    recall,
    "F1":        f1,
    "Support":   support.astype(int),
})

print(f"\n{'═'*60}")
print(f"  4.2 — {MODEL_A} como ground truth × {MODEL_B}")
print(f"{'═'*60}")
print(f"\n  Accuracy : {acc:.4f}\n")
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  F1-macro    : {f1_macro:.4f}")
print(f"  F1-weighted : {f1_w:.4f}")

# ── Destaque NEG ─────────────────────────────────────────────
neg_idx    = CLASS_ORDER.index("NEG")
neg_recall = recall[neg_idx]

print(f"\n{'─'*60}")
print(f"  ⚠  Recall NEG = {neg_recall:.4f}")
print(f"""
  No domínio de saúde mental, falhar em identificar um comentário
  negativo (não-suporte, hostil ou minimizador) é especialmente
  custoso: o modelo deixa passar conteúdo que pode expor usuários
  vulneráveis a respostas prejudiciais sem sinalização.
  Um Recall NEG baixo significa que o BERTimbau sistematicamente
  subestima a hostilidade — risco direto em aplicações de moderação
  ou triagem de conteúdo em comunidades como r/desabafos.
""")
print(f"{'─'*60}")


# ════════════════════════════════════════════════════════════
# 4.2.2  Matriz de confusão
# ════════════════════════════════════════════════════════════

cm = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)

# normalizada por linha (recall por célula)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Absoluta ─────────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[0], annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title("Contagens absolutas", fontsize=12, fontweight="bold")
axes[0].set_xlabel(f"Predito — {MODEL_B}", fontsize=10)
axes[0].set_ylabel(f"Real — {MODEL_A}", fontsize=10)

# destaca diagonal
for i in range(len(CLASS_ORDER)):
    axes[0].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

# ── Normalizada (recall) ─────────────────────────────────────
annot_norm = np.array([[f"{v:.2f}" for v in row] for row in cm_norm])

sns.heatmap(cm_norm, annot=annot_norm, fmt="", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[1], annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title("Normalizada por linha (Recall)", fontsize=12, fontweight="bold")
axes[1].set_xlabel(f"Predito — {MODEL_B}", fontsize=10)
axes[1].set_ylabel(f"Real — {MODEL_A}", fontsize=10)

# destaca linha NEG inteira para chamar atenção
neg_row = CLASS_ORDER.index("NEG")
axes[1].add_patch(plt.Rectangle((0, neg_row), len(CLASS_ORDER), 1,
                                  fill=False, edgecolor="#E05C5C", linewidth=2.5))
# destaca diagonal
for i in range(len(CLASS_ORDER)):
    axes[1].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

fig.suptitle(f"Matriz de Confusão — {MODEL_A} (ground truth) × {MODEL_B}",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sec4_02_matriz_confusao.png")
plt.show()

#### 3.8 - Simbólico - BERTimbau

In [ ]:
# ============================================================
# SEÇÃO 5 — Simbólico × BERTimbau
# ============================================================

MODEL_A = "Symbolic"
MODEL_B = "BERTimbau"

COL_A = OUTPUT_COLS[MODEL_A]
COL_B = OUTPUT_COLS[MODEL_B]

df["_5_severity"] = df.apply(lambda r: get_severity(r[COL_A], r[COL_B]), axis=1)
df_discord = df[df["_5_severity"] != "concordância"].copy()
df_agree   = df[df["_5_severity"] == "concordância"].copy()


# ════════════════════════════════════════════════════════════
# 5.1  Matriz de concordância 3×3
# ════════════════════════════════════════════════════════════

conf_matrix = pd.crosstab(
    df[COL_A], df[COL_B],
    rownames=[MODEL_A], colnames=[MODEL_B]
).reindex(index=CLASS_ORDER, columns=CLASS_ORDER, fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
            linewidths=0.5, linecolor="white",
            square=True, ax=ax,
            annot_kws={"size": 13, "weight": "bold"})

ax.set_title(f"Matriz de Concordância\n{MODEL_A} × {MODEL_B}",
             fontsize=13, fontweight="bold")
ax.set_xlabel(MODEL_B, fontsize=11)
ax.set_ylabel(MODEL_A, fontsize=11)

for i in range(len(CLASS_ORDER)):
    ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                edgecolor="#2E7D32", linewidth=2.5))

plt.tight_layout()
plt.savefig("sec5_01_matriz_concordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 5.2  Taxa de concordância total e por classe
# ════════════════════════════════════════════════════════════

total_agree = (df[COL_A] == df[COL_B]).mean() * 100
print(f"\n── Taxa de concordância total: {total_agree:.2f}%\n")
print(f"── Por classe (amostras onde {MODEL_A} predisse a classe):")

for cls in CLASS_ORDER:
    mask      = df[COL_A] == cls
    agree_cls = (df.loc[mask, COL_A] == df.loc[mask, COL_B]).mean() * 100
    n         = mask.sum()
    print(f"   {cls}: {agree_cls:.2f}%  (n={n})")


# ════════════════════════════════════════════════════════════
# 5.3  Proporção de discordâncias leves vs. severas
# ════════════════════════════════════════════════════════════

sev_counts = df_discord["_5_severity"].value_counts()
n_leve     = sev_counts.get("leve",   0)
n_severa   = sev_counts.get("severa", 0)
n_discord  = len(df_discord)

print(f"\n── Discordâncias totais: {n_discord} ({n_discord/len(df)*100:.1f}% das amostras)")
print(f"   Leves  (adjacentes): {n_leve}  ({n_leve/n_discord*100:.1f}%)")
print(f"   Severas (POS↔NEG):   {n_severa} ({n_severa/n_discord*100:.1f}%)")

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(["Leve\n(adjacente)", "Severa\n(POS↔NEG)"],
              [n_leve/n_discord*100, n_severa/n_discord*100],
              color=["#F5A623", "#E05C5C"], edgecolor="white", width=0.45)

for bar, val in zip(bars, [n_leve/n_discord*100, n_severa/n_discord*100]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("% das discordâncias")
ax.set_ylim(0, max(n_leve, n_severa)/n_discord*100 * 1.25)
ax.set_title(f"Tipo de Discordância\n{MODEL_A} × {MODEL_B}",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("sec5_03_tipo_discordancia.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 5.4  Boxplot da confiança do BERTimbau: concorda vs. discorda
# ════════════════════════════════════════════════════════════

# confiança do BERTimbau na sua própria predição
for subset, label in [(df_agree, "agree"), (df_discord, "discord")]:
    subset[f"_conf_{MODEL_B}"] = subset.apply(
        lambda r: r[CONF_COLS[MODEL_B][r[COL_B]]], axis=1)

conf_agree   = df_agree[f"_conf_{MODEL_B}"].dropna()
conf_discord = df_discord[f"_conf_{MODEL_B}"].dropna()

fig, ax = plt.subplots(figsize=(7, 5))

bp = ax.boxplot([conf_agree, conf_discord],
                patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=2),
                flierprops=dict(marker=".", markersize=2, alpha=0.3))

colors = ["#4CAF82", "#E05C5C"]
labels = [f"Concordam\n(n={len(conf_agree)})",
          f"Discordam\n(n={len(conf_discord)})"]

for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticks([1, 2])
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel(f"Confiança — {MODEL_B}")
ax.set_ylim(0, 1.05)
ax.set_title(f"Confiança do {MODEL_B}\nConcordância vs. Discordância com {MODEL_A}",
             fontsize=12, fontweight="bold")

# anota medianas
for i, data in enumerate([conf_agree, conf_discord], 1):
    ax.text(i, data.median() + 0.02, f"md={data.median():.2f}",
            ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("sec5_04_boxplot_confianca_bertimbau.png")
plt.show()


# ════════════════════════════════════════════════════════════
# 5.5  Exemplos de discordância explicáveis pelas regras simbólicas
# ════════════════════════════════════════════════════════════

# confiança do BERTimbau na sua própria predição
df_discord["_conf_B_5"] = df_discord.apply(
    lambda r: r[CONF_COLS[MODEL_B][r[COL_B]]], axis=1)

df_discord_exp = (
    df_discord
    # apenas discordâncias severas POS↔NEG (sem neutro)
    .loc[df_discord["_5_severity"] == "severa"]
    .assign(_abs_score=lambda d: d["Symbolic_Score"].abs())
    # 1º critério: maior evidência léxica; 2º: maior confiança do BERTimbau
    .sort_values(["_abs_score", "_conf_B_5"], ascending=[False, False])
)

N_EXAMPLES = 5
print(f"\n{'═'*70}")
print(f"5.5 — EXEMPLOS DE DISCORDÂNCIA SEVERA (POS↔NEG)")
print(f"      ordenados por |Symbolic_Score| desc, confiança BERTimbau desc")
print(f"{'═'*70}")

for i, (_, row) in enumerate(df_discord_exp.head(N_EXAMPLES).iterrows(), 1):
    body_preview = str(row["body"])[:250] + ("..." if len(str(row["body"])) > 250 else "")
    terms        = row["Symbolic_Terms"] if isinstance(row["Symbolic_Terms"], list) \
                   else str(row["Symbolic_Terms"])

    print(f"\n  [{i}] {body_preview}")
    print(f"       → Symbolic : {row[COL_A]}  (score={row['Symbolic_Score']:+.2f})")
    print(f"       → {MODEL_B}: {row[COL_B]}  (confiança={row['_conf_B_5']:.2f})")
    print(f"       → Termos simbólicos: {terms}")

print(f"\n{'═'*70}")

#### 3.9 - Ensemble Pysentimiento + XLM-RoBERTa como Ground Truth

In [ ]:
# ============================================================
# SEÇÃO 6 — Ensemble Pysentimiento + XLM-RoBERTa como Ground Truth
# ============================================================

from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, precision_recall_fscore_support,
                              cohen_kappa_score)

COL_PYS  = OUTPUT_COLS["Pysentimiento"]
COL_XLM  = OUTPUT_COLS["XLM-RoBERTa"]
COL_BERT = OUTPUT_COLS["BERTimbau"]


# ════════════════════════════════════════════════════════════
# 6.1  Cohen's Kappa entre Pysentimiento e XLM-RoBERTa
#      (justifica a qualidade do ensemble antes de usá-lo)
# ════════════════════════════════════════════════════════════

kappa = cohen_kappa_score(df[COL_PYS], df[COL_XLM])

print(f"\n{'═'*60}")
print(f"  6.1 — Cohen's Kappa: Pysentimiento × XLM-RoBERTa")
print(f"{'═'*60}")
print(f"\n  κ = {kappa:.4f}")
print(f"""
  Interpretação (Landis & Koch):
    κ < 0.20  → concordância quase nula
    0.20–0.40 → razoável
    0.40–0.60 → moderada
    0.60–0.80 → substancial
    0.80–1.00 → quase perfeita
""")
print(f"{'═'*60}")


# ════════════════════════════════════════════════════════════
# 6.2  Filtragem — manter apenas os casos de concordância
# ════════════════════════════════════════════════════════════

mask_agree  = df[COL_PYS] == df[COL_XLM]
df_ensemble = df[mask_agree].copy()
df_ensemble["_ensemble_label"] = df_ensemble[COL_PYS]  # mesmos por definição

n_total     = len(df)
n_kept      = len(df_ensemble)
n_discarded = n_total - n_kept

print(f"\n{'═'*60}")
print(f"  6.2 — Filtragem do dataset")
print(f"{'═'*60}")
print(f"\n  Total de amostras    : {n_total}")
print(f"  Mantidas (concordam) : {n_kept}  ({n_kept/n_total*100:.1f}%)")
print(f"  Descartadas          : {n_discarded}  ({n_discarded/n_total*100:.1f}%)")
print(f"\n  Distribuição do ensemble (subconjunto filtrado):")
for cls in CLASS_ORDER:
    n_cls = (df_ensemble["_ensemble_label"] == cls).sum()
    print(f"    {cls}: {n_cls}  ({n_cls/n_kept*100:.1f}%)")
print(f"{'═'*60}")


# ════════════════════════════════════════════════════════════
# 6.3  Gráfico de barras — amostras mantidas vs. descartadas
# ════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(["Mantidas\n(concordam)", "Descartadas\n(divergem)"],
              [n_kept / n_total * 100, n_discarded / n_total * 100],
              color=["#4CAF82", "#E05C5C"], edgecolor="white", width=0.45)

for bar, val in zip(bars, [n_kept / n_total * 100, n_discarded / n_total * 100]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=11, fontweight="bold")

ax.set_ylabel("% do total de amostras")
ax.set_ylim(0, max(n_kept, n_discarded) / n_total * 100 * 1.25)
ax.set_title("Filtro do Ensemble\nPysentimiento × XLM-RoBERTa",
             fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("sec6_01_filtro_ensemble.png")
plt.show()
print("✔  6.3 — Gráfico de filtro do ensemble salvo.")


# ════════════════════════════════════════════════════════════
# 6.4  Métricas do BERTimbau vs. ensemble (ground truth)
# ════════════════════════════════════════════════════════════

y_true = df_ensemble["_ensemble_label"]
y_pred = df_ensemble[COL_BERT]

acc = accuracy_score(y_true, y_pred)

precision, recall, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=CLASS_ORDER, zero_division=0)

f1_macro = f1.mean()
_, _, f1_w, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame({
    "Classe":    CLASS_ORDER,
    "Precision": precision,
    "Recall":    recall,
    "F1":        f1,
    "Support":   support.astype(int),
})

print(f"\n{'═'*60}")
print(f"  6.4 — Ensemble (Pysentimiento + XLM-RoBERTa) como ground truth × BERTimbau")
print(f"{'═'*60}")
print(f"\n  Accuracy : {acc:.4f}\n")
print(metrics_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(f"\n  F1-macro    : {f1_macro:.4f}")
print(f"  F1-weighted : {f1_w:.4f}")

# ── Destaque NEG ─────────────────────────────────────────────
neg_idx    = CLASS_ORDER.index("NEG")
neg_recall = recall[neg_idx]

print(f"\n{'─'*60}")
print(f"  ⚠  Recall NEG = {neg_recall:.4f}")
print(f"""
  No domínio de saúde mental, o ensemble filtra os casos de maior
  ambiguidade (onde Pysentimiento e XLM-RoBERTa divergem), tornando
  o ground truth mais confiável — mas também mais exigente.
  Um Recall NEG baixo nesse subconjunto é ainda mais preocupante:
  o BERTimbau falha justamente nos textos negativos sobre os quais
  dois modelos independentes concordavam plenamente, indicando uma
  limitação estrutural do fine-tuning e não apenas ruído de rótulo.
  Em aplicações de moderação ou triagem em r/desabafos e
  r/AnsiedadeDepressao, essa lacuna representa risco real de deixar
  conteúdo hostil ou minimizador sem sinalização.
""")
print(f"{'─'*60}")


# ════════════════════════════════════════════════════════════
# 6.5  Matriz de confusão do BERTimbau vs. ensemble
# ════════════════════════════════════════════════════════════

cm      = confusion_matrix(y_true, y_pred, labels=CLASS_ORDER)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Absoluta ─────────────────────────────────────────────────
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[0], annot_kws={"size": 13, "weight": "bold"})
axes[0].set_title("Contagens absolutas", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predito — BERTimbau", fontsize=10)
axes[0].set_ylabel("Real — Ensemble", fontsize=10)

for i in range(len(CLASS_ORDER)):
    axes[0].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

# ── Normalizada (recall) ─────────────────────────────────────
annot_norm = np.array([[f"{v:.2f}" for v in row] for row in cm_norm])

sns.heatmap(cm_norm, annot=annot_norm, fmt="", cmap="Blues",
            xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
            vmin=0, vmax=1,
            linewidths=0.5, linecolor="white", square=True,
            ax=axes[1], annot_kws={"size": 13, "weight": "bold"})
axes[1].set_title("Normalizada por linha (Recall)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Predito — BERTimbau", fontsize=10)
axes[1].set_ylabel("Real — Ensemble", fontsize=10)

# destaca linha NEG inteira
neg_row = CLASS_ORDER.index("NEG")
axes[1].add_patch(plt.Rectangle((0, neg_row), len(CLASS_ORDER), 1,
                                  fill=False, edgecolor="#E05C5C", linewidth=2.5))
# destaca diagonal
for i in range(len(CLASS_ORDER)):
    axes[1].add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                     edgecolor="#2E7D32", linewidth=2.5))

fig.suptitle("Matriz de Confusão — Ensemble como Ground Truth × BERTimbau",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("sec6_02_matriz_confusao_ensemble.png")
plt.show()